In [1]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [146]:
df = pd.read_csv(r'/home/user/Downloads/DataHut_IE_Dunnes_FullDump_20261106.CSV', low_memory=False, sep='|')

In [147]:
df.shape

(30863, 132)

In [4]:
import pandas as pd
import re



print("Total records:", len(df))
print("Total columns:", len(df.columns))

issues = []

# Check text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns

for col in text_cols:

    s = df[col].astype(str)

    # Leading/trailing whitespace
    mask = s.str.contains(r"^\s|\s$", regex=True)

    # Multiple consecutive spaces
    mask |= s.str.contains(r" {2,}", regex=True)

    # Tab characters
    mask |= s.str.contains(r"\t", regex=True)

    # Newline / carriage return
    mask |= s.str.contains(r"[\r\n]", regex=True)

    # Unescaped/control characters
    mask |= s.str.contains(
        r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]",
        regex=True
    )

    if mask.any():

        # Only the FIRST issue from this column
        idx = df.index[mask][0]

        issues.append({
            "row_no": idx + 2,
            "url": df.loc[idx, "pdp_url"],
            "column": col
        })


# Display results
if issues:

    result = pd.DataFrame(issues)

    print("\n==============================")
    print("Total columns with issues:", len(result))
    print("==============================\n")

    print(result.to_string(index=False))

else:

    print("\nNo whitespace, extra space, newline, tab, or unescaped character issues found.")

Total records: 11285
Total columns: 132

Total columns with issues: 1

 row_no                                                                                                                                   url                  column
    791 https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-fresh-irish-gourmet-sirloin-beef-burgers-2-pack-260-g-id-1574099000 nutritional_information


In [5]:
import pandas as pd

# Promotion date columns
date_cols = ["promotion_valid_from", "promotion_valid_upto"]

# Convert dates without modifying original data
dates = df[date_cols].apply(
    lambda col: pd.to_datetime(col, errors="coerce", format="mixed")
)

today = pd.Timestamp.today().normalize()

# 1. Invalid date formats (excluding empty values)
for col in date_cols:
    original = df[col].astype("string").str.strip()

    mask = (
        original.notna()
        & original.ne("")
        & dates[col].isna()
    )

    result = df.loc[mask, ["pdp_url", col]].copy()
    result.insert(0, "row_no", result.index + 2)

    print(f"\nInvalid dates in {col}: {len(result)}")
    print(result.head(20).to_string(index=False))


# 2. Future promotion start dates
mask = dates["promotion_valid_from"] > today

result = df.loc[
    mask, ["pdp_url", "promotion_valid_from", "promotion_valid_upto"]
].copy()

result.insert(0, "row_no", result.index + 2)

print("\nFuture promotion start dates:", len(result))
print(result.head(20).to_string(index=False))


# 3. Future promotion end dates
mask = dates["promotion_valid_upto"] > today

result = df.loc[
    mask, ["pdp_url", "promotion_valid_from", "promotion_valid_upto"]
].copy()

result.insert(0, "row_no", result.index + 2)

print("\nFuture promotion end dates:", len(result))
print(result.head(20).to_string(index=False))


# 4. Promotion start date greater than end date
mask = (
    dates["promotion_valid_from"].notna()
    & dates["promotion_valid_upto"].notna()
    & (dates["promotion_valid_from"] > dates["promotion_valid_upto"])
)

result = df.loc[
    mask, ["pdp_url", "promotion_valid_from", "promotion_valid_upto"]
].copy()

result.insert(0, "row_no", result.index + 2)

print("\nInvalid promotion date ranges:", len(result))
print(result.head(20).to_string(index=False))


# 5. Overall promotion date ranges
print("\nPromotion date ranges:")

for col in date_cols:
    print(
        f"{col}:",
        dates[col].min(),
        "to",
        dates[col].max()
    )


Invalid dates in promotion_valid_from: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_from]
Index: []

Invalid dates in promotion_valid_upto: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_upto]
Index: []

Future promotion start dates: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_from, promotion_valid_upto]
Index: []

Future promotion end dates: 3607
 row_no                                                                                                                             pdp_url promotion_valid_from promotion_valid_upto
     11                  https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-blueberries-225-g-id-1683185003           22/10/2026           12/11/2026
     13                       https://shop.supervalu.ie/sm/delivery/rsid/5550/product/keelings-strawberries-family-pack-350-g-id-1386984000           22/10/2026           12/11/2026
     23                      https://shop.supervalu.ie/s

In [6]:
df.columns

Index(['unique_id', 'competitor_name', 'store_name', 'store_addressline1',
       'store_addressline2', 'store_suburb', 'store_state', 'store_postcode',
       'store_addressid', 'extraction_date',
       ...
       'product_unique_key', 'multibuy_items_pricesingle', 'perfect_match',
       'servings_per_pack', 'warning', 'suitable_for', 'standard_drinks',
       'environmental', 'grape_variety', 'retail_limit'],
      dtype='object', length=132)

In [7]:
import pandas as pd


# Check missing or unwanted IDs
unwanted_values = ["NULL", "NAN", "NONE", "N/A", "NA", "UNDEFINED"]

id_text = df["unique_id"].astype("string").str.strip()

mask = (
    id_text.isna()
    | id_text.eq("")
    | id_text.str.upper().isin(unwanted_values)
)

result = df.loc[mask, ["unique_id", "pdp_url"]].copy()
result["row_no"] = result.index + 2

print("Missing/unwanted IDs:", len(result))
print(result[["row_no", "unique_id", "pdp_url"]].to_string(index=False))

# Check duplicate IDs
duplicate_mask = (
    id_text.notna()
    & id_text.ne("")
    & ~id_text.str.upper().isin(unwanted_values)
    & id_text.duplicated(keep=False)
)

duplicates = df.loc[duplicate_mask, ["unique_id", "pdp_url"]].copy()
duplicates["row_no"] = duplicates.index + 2

print("\nDuplicate ID records:", len(duplicates))
print(
    duplicates[["row_no", "unique_id", "pdp_url"]]
    .sort_values("unique_id")
    .to_string(index=False)
)

Missing/unwanted IDs: 0
Empty DataFrame
Columns: [row_no, unique_id, pdp_url]
Index: []

Duplicate ID records: 0
Empty DataFrame
Columns: [row_no, unique_id, pdp_url]
Index: []


In [8]:
import pandas as pd

value = df["unique_id"].fillna("").astype(str).str.strip()

mask = value.ne("") & ~value.str.fullmatch(r"\d+", na=False)

result = df.loc[mask, ["pdp_url", "unique_id"]].copy()
result["record_no"] = result.index + 2
result["field"] = "unique_id"

print(result[["record_no", "pdp_url", "field", "unique_id"]].to_string(index=False))

Empty DataFrame
Columns: [record_no, pdp_url, field, unique_id]
Index: []


In [151]:
req_cols = """unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,
site_shown_uom

"""

req_cols = [c.strip() for c in req_cols.split(",") if c.strip()]

missing = [c for c in req_cols if c not in df.columns]
extra = [c for c in df.columns if c not in req_cols]

print("Requirement columns" 
":", len(req_cols))
print("File columns:", len(df.columns))
print("Missing:", missing)
print("Extra:", extra)
print("Order matches:", req_cols == list(df.columns))

Requirement columns: 21
File columns: 132
Missing: ['pack_size']
Extra: ['store_name', 'store_addressline1', 'store_addressline2', 'store_suburb', 'store_state', 'store_postcode', 'store_addressid', 'brand', 'brand_type', 'drained_weight', 'producthierarchy_level1', 'producthierarchy_level2', 'producthierarchy_level3', 'producthierarchy_level4', 'producthierarchy_level5', 'producthierarchy_level6', 'producthierarchy_level7', 'price_was', 'promotion_type', 'package_sizeof_sellingprice', 'per_unit_sizedescription', 'multi_buy_item_count', 'multi_buy_items_price_total', 'variants', 'product_description', 'instructions', 'storage_instructions', 'preparationinstructions', 'instructionforuse', 'country_of_origin', 'allergens', 'age_of_the_product', 'age_recommendations', 'flavour', 'nutritions', 'nutritional_information', 'vitamins', 'labelling', 'grade', 'packaging', 'receipies', 'processed_food', 'barcode', 'frozen', 'chilled', 'organictype', 'cooking_part', 'handmade', 'max_heating_temper

In [9]:
# Empty columns as per requirement
req_empty = """ """

req_empty = [c.strip() for c in req_empty.split(",") if c.strip()]

# Empty columns in the file
file_empty = df.columns[df.isna().all()].tolist()

# Compare
missing_empty = [c for c in req_empty if c not in file_empty]
unexpected_empty = [c for c in file_empty if c not in req_empty]
matching_empty = [c for c in req_empty if c in file_empty]

print("Requirement empty columns :", len(req_empty))
print("File empty columns        :", len(file_empty))
print("Matching                 :", len(matching_empty))
print("Expected empty but not empty:", missing_empty)
print("Unexpected empty columns     :", unexpected_empty)

Requirement empty columns : 0
File empty columns        : 89
Matching                 : 0
Expected empty but not empty: []
Unexpected empty columns     : ['store_name', 'store_addressline1', 'store_addressline2', 'store_suburb', 'store_state', 'store_postcode', 'store_addressid', 'brand', 'brand_type', 'drained_weight', 'producthierarchy_level5', 'producthierarchy_level6', 'producthierarchy_level7', 'promotion_type', 'package_sizeof_sellingprice', 'per_unit_sizedescription', 'multi_buy_item_count', 'multi_buy_items_price_total', 'variants', 'instructions', 'preparationinstructions', 'age_of_the_product', 'age_recommendations', 'flavour', 'nutritions', 'vitamins', 'labelling', 'grade', 'region', 'packaging', 'receipies', 'processed_food', 'frozen', 'chilled', 'organictype', 'cooking_part', 'handmade', 'max_heating_temperature', 'special_information', 'label_information', 'dimensions', 'special_nutrition_purpose', 'feeding_recommendation', 'warranty', 'color', 'model_number', 'material',

In [10]:
# Check invalid price values
# Digits, comma and decimal point are allowed

price = df["price"].fillna("").astype(str).str.strip()

mask = (
    price.ne("")
    & ~price.str.fullmatch(r"\d+(,\d+)*(\.\d+)?", na=False)
)

result = df.loc[mask, ["url", "price"]].copy()
result["record_no"] = result.index + 2

print("Invalid price values:", len(result))
print(
    result[
        ["record_no", "url", "price"]
    ].to_string(index=False)
)

KeyError: 'price'

In [11]:
df[df.currency.isna()].price.value_counts()

AttributeError: 'DataFrame' object has no attribute 'price'

In [12]:
df.currency.value_counts()

currency
EUR    11285
Name: count, dtype: int64

In [13]:
df.competitor_name.value_counts()

competitor_name
supervalu    11285
Name: count, dtype: int64

In [14]:
df.instock.value_counts()

instock
True     11219
False       66
Name: count, dtype: int64

In [15]:
df.organictype.value_counts()

Series([], Name: count, dtype: int64)

In [16]:
df.duplicated().sum()

np.int64(0)

In [17]:
df.pdp_url.duplicated().sum()

np.int64(0)

In [18]:
df.unique_id.duplicated().sum()

np.int64(0)

In [19]:
df.extraction_date.value_counts()

extraction_date
2026-11-06    11285
Name: count, dtype: int64

In [20]:
import re

value = df["company_phone"].fillna("").astype(str).str.strip()

digits = value.str.replace(r"\D", "", regex=True)

mask = (
    value.ne("")
    & (
        ~value.str.match(r"^[0-9+().\-\s]+$", na=False)
        | (digits.str.len() < 7)
    )
)

result = df.loc[mask, ["url", "company_phone"]].copy()
result["record_no"] = result.index + 2
result["field"] = "company_phone"

print(result[["record_no", "url", "field", "company_phone"]].to_string(index=False))

KeyError: 'company_phone'

In [21]:
text_cols = [
    "main_category",
    "category",
    "sub_category",
    "company_name",
    "company_location",
    "contact_person"
]

for col in text_cols:

    value = df[col].fillna("").astype(str).str.strip()

    mask = value.ne("") & value.str.fullmatch(r"\d+", na=False)

    result = df.loc[mask, ["url", col]].copy()
    result["record_no"] = result.index + 2
    result["field"] = col

    print(f"\nInvalid {col}: {len(result)}")
    print(
        result[["record_no", "url", "field", col]]
        .head(5)
        .to_string(index=False)
    )

KeyError: 'main_category'

In [22]:
value = df["spec_fields"].fillna("").astype(str).str.strip()

mask = value.str.contains(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", regex=True, na=False)

result = df.loc[mask, ["url", "spec_fields"]].copy()
result["record_no"] = result.index + 2
result["field"] = "spec_fields"

print(result[["record_no", "url", "field", "spec_fields"]].head(5).to_string(index=False))

KeyError: 'spec_fields'

In [23]:
df.isnull().sum()

unique_id                          0
competitor_name                    0
store_name                     11285
store_addressline1             11285
store_addressline2             11285
store_suburb                   11285
store_state                    11285
store_postcode                 11285
store_addressid                11285
extraction_date                    0
product_name                       0
brand                          11285
brand_type                     11285
grammage_quantity                  0
grammage_unit                      0
drained_weight                 11285
producthierarchy_level1            0
producthierarchy_level2            0
producthierarchy_level3          738
producthierarchy_level4         9224
producthierarchy_level5        11285
producthierarchy_level6        11285
producthierarchy_level7        11285
regular_price                      0
selling_price                      0
price_was                       8716
promotion_price                 8716
p

In [25]:
df[df.product_description.isna()].pdp_url.head()

Series([], Name: pdp_url, dtype: object)

In [24]:
import pandas as pd

# ============================================================
# 1. CHECK INVALID unique_id VALUES
# ============================================================

unique_id = df["unique_id"].fillna("").astype(str).str.strip()

# Invalid if anything other than digits is present
invalid_mask = ~unique_id.str.fullmatch(r"\d+")

invalid_result = df.loc[
    invalid_mask,
    ["unique_id", "pdp_url"]
].copy()

invalid_result["row_no"] = invalid_result.index + 2

invalid_result = invalid_result[
    ["row_no", "pdp_url", "unique_id"]
]


# ============================================================
# 2. CHECK DUPLICATE unique_id VALUES
# ============================================================

duplicate_mask = df["unique_id"].duplicated(keep=False)

duplicate_result = df.loc[
    duplicate_mask,
    ["unique_id", "pdp_url"]
].copy()

duplicate_result["row_no"] = duplicate_result.index + 2

duplicate_result = duplicate_result[
    ["row_no", "pdp_url", "unique_id"]
]


# ============================================================
# 3. PRINT INVALID IDs
# ============================================================

print("\n==============================")
print("Invalid unique_id")
print("==============================")

print("Total invalid unique_id:", len(invalid_result))

if len(invalid_result) > 0:
    print(invalid_result.to_string(index=False))
else:
    print("No invalid unique_id found.")


# ============================================================
# 4. PRINT DUPLICATE IDs
# ============================================================

print("\n==============================")
print("Duplicate unique_id")
print("==============================")

print("Total duplicate records:", len(duplicate_result))

if len(duplicate_result) > 0:
    print(duplicate_result.to_string(index=False))
else:
    print("No duplicate unique_id found.")


# ============================================================
# 5. SUMMARY
# ============================================================

print("\n==============================")
print("Summary")
print("==============================")

print("Total records:", len(df))
print("Unique unique_id count:", df["unique_id"].nunique())
print("Invalid unique_id:", len(invalid_result))
print("Duplicate records:", len(duplicate_result))


Invalid unique_id
Total invalid unique_id: 0
No invalid unique_id found.

Duplicate unique_id
Total duplicate records: 0
No duplicate unique_id found.

Summary
Total records: 11285
Unique unique_id count: 11285
Invalid unique_id: 0
Duplicate records: 0


In [25]:
import pandas as pd

hierarchy_cols = [
    "producthierarchy_level1",
    "producthierarchy_level2",
    "producthierarchy_level3",
    "producthierarchy_level4",
    "producthierarchy_level5",
    "producthierarchy_level6",
    "producthierarchy_level7"
]

issues = []

for idx, row in df.iterrows():

    breadcrumb = str(row["breadcrumb"]).strip()

    if not breadcrumb:
        continue

    # Split breadcrumb using >
    breadcrumb_levels = [
        x.strip()
        for x in breadcrumb.split(">")
    ]

    for i, breadcrumb_value in enumerate(breadcrumb_levels):

        if i >= len(hierarchy_cols):
            break

        hierarchy_col = hierarchy_cols[i]
        hierarchy_value = row[hierarchy_col]

        # Breadcrumb has value but hierarchy is missing
        if (
            breadcrumb_value != ""
            and (
                pd.isna(hierarchy_value)
                or str(hierarchy_value).strip() == ""
            )
        ):
            issues.append({
                "row_no": idx + 2,
                "pdp_url": row["pdp_url"],
                "breadcrumb": breadcrumb,
                "missing_hierarchy": hierarchy_col,
                "breadcrumb_value": breadcrumb_value
            })

result = pd.DataFrame(issues)

print("Total issues:", len(result))

if not result.empty:
    print(result.to_string(index=False))

Total issues: 0


In [26]:
import pandas as pd

# Convert price fields to numeric
promotion_price = pd.to_numeric(
    df["promotion_price"], errors="coerce"
)

multibuy_total = pd.to_numeric(
    df["multi_buy_items_price_total"], errors="coerce"
)

selling_price = pd.to_numeric(
    df["selling_price"], errors="coerce"
)

# Normalize promotion_type
promotion_type = (
    df["promotion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Expected promotion_price:
# multibuy → multi_buy_items_price_total
# otherwise → selling_price
expected_price = multibuy_total.where(
    promotion_type == "multibuy",
    selling_price
)

# Find where promotion_price is different from expected value
mask = (
    promotion_price.notna() &
    expected_price.notna() &
    (promotion_price != expected_price)
)

issues = df.loc[
    mask,
    [
        "promotion_type",
        "promotion_price",
        "multi_buy_items_price_total",
        "selling_price",
        "pdp_url"
    ]
].copy()

# Add Excel row number
issues["row_no"] = issues.index + 2

# Arrange columns
issues = issues[
    [
        "row_no",
        "pdp_url",
        "promotion_type",
        "promotion_price",
        "multi_buy_items_price_total",
        "selling_price"
    ]
]

print(f"Total promotion_price issues: {len(issues)}")
print(issues.to_string(index=False))

Total promotion_price issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_type, promotion_price, multi_buy_items_price_total, selling_price]
Index: []


In [27]:
import pandas as pd

issues = []

# -------------------------------
# Normalize columns
# -------------------------------

df["promotion_description_clean"] = (
    df["promotion_description"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["producthierarchy_level3_clean"] = (
    df["producthierarchy_level3"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["producthierarchy_level4_clean"] = (
    df["producthierarchy_level4"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Convert prices
for col in [
    "regular_price",
    "selling_price",
    "promotion_price",
    "price_was"
]:
    df[col + "_num"] = pd.to_numeric(df[col], errors="coerce")


# ==================================================
# 1. Fleisch & Fisch validation
# ==================================================

fleisch_mask = (
    (df["producthierarchy_level3_clean"] == "fleisch & fisch") &
    (df["producthierarchy_level4_clean"] != "charcuterie & wurstwaren")
)

fleisch = df[fleisch_mask]


# price_was should be empty
mask = (
    fleisch["price_was"]
    .astype("string")
    .fillna("")
    .str.strip()
    .ne("")
)

for idx in fleisch[mask].index:
    issues.append([
        idx + 2,
        df.loc[idx, "pdp_url"],
        "Fleisch & Fisch issue",
        "price_was should be empty but value is present"
    ])


# ==================================================
# 2. HIT Promotion validation
# ==================================================

hit = df[
    df["promotion_description_clean"] == "hit"
]


for idx, row in hit.iterrows():

    regular = row["regular_price_num"]
    selling = row["selling_price_num"]
    promotion = row["promotion_price_num"]

    price_was = row["price_was_num"]

    # Single price HIT
    if pd.notna(regular) and pd.notna(selling) and regular == selling:

        if (
            promotion != selling or
            str(row["regular_price"]).strip() not in ["", "nan"] or
            str(row["price_was"]).strip() not in ["", "nan"]
        ):
            issues.append([
                idx + 2,
                row["pdp_url"],
                "HIT promotion issue",
                "Single price HIT: promotion_price should equal selling_price and regular_price/price_was should be empty"
            ])

    # Two price HIT
    elif pd.notna(regular) and pd.notna(selling) and regular != selling:

        if (
            promotion != selling or
            price_was != regular
        ):
            issues.append([
                idx + 2,
                row["pdp_url"],
                "HIT promotion issue",
                "Two price HIT: promotion_price should equal selling_price and price_was should equal regular_price"
            ])


# ==================================================
# Final output
# ==================================================

if issues:
    result = pd.DataFrame(
        issues,
        columns=[
            "row_no",
            "pdp_url",
            "issue_type",
            "issue_description"
        ]
    )

    print(f"Total issues found: {len(result)}")
    print(result.to_string(index=False))

else:
    print("No issues found.")

No issues found.


In [28]:
import pandas as pd

promotion_price = pd.to_numeric(df["promotion_price"], errors="coerce")
selling_price = pd.to_numeric(df["selling_price"], errors="coerce")
multibuy_total = pd.to_numeric(
    df["multi_buy_items_price_total"], errors="coerce"
)

promotion_type = (
    df["promotion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Expected promotion price
expected_price = multibuy_total.where(
    promotion_type == "multibuy",
    selling_price
)

# Find mismatches
mask = (
    promotion_price.notna() &
    expected_price.notna() &
    (promotion_price != expected_price)
)

issues = df.loc[
    mask,
    [
        "promotion_type",
        "promotion_price",
        "selling_price",
        "multi_buy_items_price_total",
        "pdp_url"
    ]
].copy()

# Row number
issues["row_no"] = issues.index + 2

issues = issues[
    [
        "row_no",
        "pdp_url",
        "promotion_type",
        "promotion_price",
        "selling_price",
        "multi_buy_items_price_total"
    ]
]

print(f"Total promotion price issues: {len(issues)}")
print(issues.to_string(index=False))

Total promotion price issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_type, promotion_price, selling_price, multi_buy_items_price_total]
Index: []


In [29]:
import pandas as pd

desc = (
    df["promotion_description"]
    .astype("string")
    .str.strip()
    .str.lower()
)

promo_type = (
    df["promotion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Expected promotion type
expected_type = pd.Series(pd.NA, index=df.index, dtype="string")


# MultiBuy: description contains "for"
expected_type.loc[desc.str.contains(r"\bfor\b", na=False)] = "multibuy"

# % Discount: description contains was / save / % off / 1/2 price
discount_mask = desc.str.contains(
    r"\bwas\b|\bsave\b|%[\s]*off|\b1/2\s*price\b",
    regex=True,
    na=False
)

expected_type.loc[discount_mask] = "% discount"

# Find mismatches
mask = (
    expected_type.notna() &
    promo_type.notna() &
    (promo_type != expected_type)
)

issues = df.loc[
    mask,
    ["pdp_url", "promotion_description", "promotion_type"]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    ["row_no", "pdp_url", "promotion_description", "promotion_type"]
]

print(f"Total promotion_type issues: {len(issues)}")
print(issues.to_string(index=False))

Total promotion_type issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, promotion_type]
Index: []


In [30]:
import pandas as pd

# Normalize promotion type
promotion_type = (
    df["promotion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Convert prices to numeric
promotion_price = pd.to_numeric(df["promotion_price"], errors="coerce")
selling_price = pd.to_numeric(df["selling_price"], errors="coerce")

# Find issues
mask = (
    promotion_type.notna() &
    (promotion_type != "multibuy") &
    promotion_price.notna() &
    selling_price.notna() &
    (promotion_price != selling_price)
)

issues = df.loc[
    mask,
    [
        "pdp_url",
        "promotion_type",
        "promotion_price",
        "selling_price"
    ]
].copy()

# Add row number
issues["row_no"] = issues.index + 2

issues = issues[
    [
        "row_no",
        "pdp_url",
        "promotion_type",
        "promotion_price",
        "selling_price"
    ]
]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_type, promotion_price, selling_price]
Index: []


In [31]:
# Check: promotion information is present but promotion_price is missing

percentage_discount = df['percentage_discount'].fillna('').astype(str).str.strip()
promotion_description = df['promotion_description'].fillna('').astype(str).str.strip()
promotion_price = df['promotion_price'].fillna('').astype(str).str.strip()

mask = (
    (percentage_discount != '') &
    (promotion_description != '') &
    (promotion_price == '')
)

issues = df.loc[
    mask,
    ['pdp_url', 'percentage_discount', 'promotion_description', 'promotion_price']
].copy()

# Excel row number
issues['row_no'] = issues.index + 2

issues = issues[
    ['row_no', 'pdp_url', 'percentage_discount',
     'promotion_description', 'promotion_price']
]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, percentage_discount, promotion_description, promotion_price]
Index: []


In [32]:
mask = (
    df["product_unique_key"].astype(str)
    !=
    (df["unique_id"].astype(str) + "P")
)

issues = df.loc[mask, ["unique_id", "product_unique_key", "pdp_url"]].copy()

issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "unique_id", "product_unique_key"]]

issues

,row_no,pdp_url,unique_id,product_unique_key


In [33]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

issues = {}

for col in text_cols:
    mask = (
        df[col]
        .fillna("")
        .astype(str)
        .str.contains(r"(?<!\\)\|", regex=True)
    )

    if mask.any():
        temp = df.loc[mask, ["pdp_url", col]].copy()
        temp.insert(0, "row_no", temp.index)  # DataFrame row number
        issues[col] = temp

# Print results
if issues:
    print("Columns containing unescaped '|':\n")

    for col, data in issues.items():
        print(f"\n=== {col} ({len(data)} rows) ===")
        print(data.to_string(index=False))
else:
    print("No unescaped '|' found.")

No unescaped '|' found.


In [34]:
import pandas as pd

escape_pattern = r'[\n\r\t\b\f\v]'

issues = []

for col in df.select_dtypes(include="object").columns:
    mask = df[col].fillna("").astype(str).str.contains(
        escape_pattern,
        regex=True,
        na=False
    )

    if mask.any():
        temp = df.loc[mask, [col]].copy()
        temp.insert(0, "row_no", temp.index + 2)
        temp["column"] = col
        temp["issue_value"] = temp[col]

        if "pdp_url" in df.columns:
            temp["pdp_url"] = df.loc[mask, "pdp_url"]

        issues.append(
            temp[["row_no", "pdp_url", "column", "issue_value"]]
        )

if issues:
    result = pd.concat(issues, ignore_index=True)

    print("=== Escape Character Issues ===")
    print(result.to_string(index=False))
    print(f"\nTotal issues: {len(result)}")
else:
    print("No escape characters found.")

No escape characters found.


In [35]:
df.nunique()

unique_id                        11285
competitor_name                      1
store_name                           0
store_addressline1                   0
store_addressline2                   0
store_suburb                         0
store_state                          0
store_postcode                       0
store_addressid                      0
extraction_date                      1
product_name                     11251
brand                                0
brand_type                           0
grammage_quantity                  490
grammage_unit                        6
drained_weight                       0
producthierarchy_level1             18
producthierarchy_level2            138
producthierarchy_level3            466
producthierarchy_level4            128
producthierarchy_level5              0
producthierarchy_level6              0
producthierarchy_level7              0
regular_price                      671
selling_price                      651
price_was                

In [36]:
df.isnull().sum()

unique_id                            0
competitor_name                      0
store_name                       11285
store_addressline1               11285
store_addressline2               11285
store_suburb                     11285
store_state                      11285
store_postcode                   11285
store_addressid                  11285
extraction_date                      0
product_name                         0
brand                            11285
brand_type                       11285
grammage_quantity                    0
grammage_unit                        0
drained_weight                   11285
producthierarchy_level1              0
producthierarchy_level2              0
producthierarchy_level3            738
producthierarchy_level4           9224
producthierarchy_level5          11285
producthierarchy_level6          11285
producthierarchy_level7          11285
regular_price                        0
selling_price                        0
price_was                

In [253]:
# private_label_brands = [
#     "Green Choice",
#     "Coles",
#     # "Woofin Good",
#     "KOi",
#     "Im",
#     "Natures Kitchen",
#     "Wellness Road"
# ]

# issues = df[
#     df['brand'].isin(private_label_brands) &
#     (df['brand_type'] != 'Private Label')
# ]

# issues[['brand', 'brand_type', 'product_name', 'pdp_url']]

In [37]:
df.dtypes

unique_id                                 int64
competitor_name                          object
store_name                              float64
store_addressline1                      float64
store_addressline2                      float64
store_suburb                            float64
store_state                             float64
store_postcode                          float64
store_addressid                         float64
extraction_date                          object
product_name                             object
brand                                   float64
brand_type                              float64
grammage_quantity                        object
grammage_unit                            object
drained_weight                          float64
producthierarchy_level1                  object
producthierarchy_level2                  object
producthierarchy_level3                  object
producthierarchy_level4                  object
producthierarchy_level5                 

In [38]:
df.promotion_type.value_counts()

Series([], Name: count, dtype: int64)

In [39]:
dup = df[df.duplicated(
    subset=['unique_id', 'store_name', 'store_postcode'],
    keep=False
)].sort_values(
    ['store_name', 'store_postcode', 'unique_id']
)

dup[['unique_id', 'store_name', 'store_postcode', 'pdp_url']].assign(
    row_no=lambda x: x.index + 2
)

,unique_id,store_name,store_postcode,pdp_url,row_no


In [40]:
df.random_weight_flag.value_counts()

Series([], Name: count, dtype: int64)

In [41]:
pattern = r'^[^>]+( > [^>]+)*$'

# Invalid breadcrumbs
invalid = df[
    df["breadcrumb"].isna() |
    (df["breadcrumb"].str.strip() == "") |
    (~df["breadcrumb"].str.fullmatch(pattern, na=False))
]

print(f"Total invalid breadcrumbs: {len(invalid)}")

if not invalid.empty:
    print(invalid[["breadcrumb"]])

Total invalid breadcrumbs: 0


In [42]:
import pandas as pd
import re

price_columns = [
    "grammage_quantity",
    "regular_price",
    "selling_price",
    "price_was",
    "promotion_price",
    "percentage_discount",
    "price_per_unit",
    "multi_buy_items_price_total",
    "multibuy_items_pricesingle"
]

for col in price_columns:

    if col not in df.columns:
        continue

    s = df[col].fillna("").astype(str).str.strip()

    # Extract only numeric-looking part
    # This allows values such as:
    # KG. za 3.58 €
    # KG. za 0.9 €
    number = s.str.extract(r"(\d[\d.,]*)", expand=False).fillna("")

    # --------------------------------------------------
    # 1. MULTIPLE DECIMAL POINTS
    # --------------------------------------------------
    multiple_decimal = number.str.count(r"\.") > 1

    # --------------------------------------------------
    # 2. COMMA USED INSTEAD OF DECIMAL POINT
    # --------------------------------------------------
    comma_decimal = number.str.contains(
        r",",
        regex=True,
        na=False
    )

    # --------------------------------------------------
    # 3. EXACTLY 2 DECIMAL PLACES
    # ONLY regular_price and selling_price
    # --------------------------------------------------
    two_decimal_issue = pd.Series(False, index=df.index)

    if col in ["regular_price", "selling_price"]:
        two_decimal_issue = (
            (s != "") &
            (~s.str.fullmatch(r"\d+\.\d{2}", na=False))
        )

    print(f"\n{'='*15} {col} {'='*15}")

    if multiple_decimal.any():
        print(f"Multiple decimal points: {multiple_decimal.sum()}")
        print(
            df.loc[multiple_decimal, [col]]
            .head(10)
            .to_string(index=False)
        )

    if comma_decimal.any():
        print(f"Comma used instead of dot: {comma_decimal.sum()}")
        print(
            df.loc[comma_decimal, [col]]
            .head(10)
            .to_string(index=False)
        )

    if two_decimal_issue.any():
        print(f"Not in 2-decimal format: {two_decimal_issue.sum()}")
        print(
            df.loc[two_decimal_issue, [col]]
            .head(10)
            .to_string(index=False)
        )


=============== grammage_quantity ===============

=============== regular_price ===============
Not in 2-decimal format: 4180
 regular_price
           5.0
           4.5
           4.0
           3.0
           4.0
           4.5
           4.5
           4.0
           4.0
           4.0

=============== selling_price ===============
Not in 2-decimal format: 5021
 selling_price
           5.0
           4.5
           4.0
           3.0
           4.0
           4.5
           4.5
           4.0
           4.0
           4.0

=============== price_was ===============

=============== promotion_price ===============

=============== percentage_discount ===============

=============== price_per_unit ===============

=============== multi_buy_items_price_total ===============

=============== multibuy_items_pricesingle ===============


In [43]:
import pandas as pd

price_columns = [
    "grammage_quantity",
    "regular_price",
    "selling_price",
    "price_was",
    "promotion_price",
    "percentage_discount",
    "price_per_unit",
    "multi_buy_items_price_total",
    "multibuy_items_pricesingle"
]

for col in price_columns:
    if col not in df.columns:
        continue

    s = df[col].fillna("").astype(str)

    issues = {
        "Zero values": pd.to_numeric(s, errors="coerce") == 0,
        "Negative values": pd.to_numeric(s, errors="coerce") < 0,
        "Multiple decimal points": s.str.count(r"\.") > 1,
        "More than 2 decimal places": s.str.contains(r"\.\d{3,}$", regex=True),
        "Non-numeric values": (~s.str.match(r"^-?\d+(\.\d+)?$", na=False)) & (s != ""),
        "Leading/Trailing whitespace": s != s.str.strip(),
    }

    print(f"\n{'='*15} {col} {'='*15}")

    for issue, mask in issues.items():
        if mask.any():
            print(f"{issue}: {mask.sum()}")
            print(df.loc[mask, [col]].head(5).to_string(index=False))


=============== grammage_quantity ===============
More than 2 decimal places: 19
grammage_quantity
            1.836
            1.116
            1.953
            1.953
            1.485
Non-numeric values: 1
grammage_quantity
            6 x 4

=============== regular_price ===============

=============== selling_price ===============

=============== price_was ===============

=============== promotion_price ===============

=============== percentage_discount ===============

=============== price_per_unit ===============
Non-numeric values: 10710
price_per_unit
      €1.82/kg
      €5.00/kg
    €0.43 each
    €0.40 each
      €3.90/kg

=============== multi_buy_items_price_total ===============

=============== multibuy_items_pricesingle ===============


In [32]:
df[df.percentage_discount == 0.0].head(3)

,unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,site_shown_uom


In [44]:
mask = (
    df["price_valid_from"].notna() &
    df["promotion_valid_from"].notna() &
    (df["price_valid_from"] != df["promotion_valid_from"])
)

issues = df.loc[
    mask,
    ["pdp_url", "price_valid_from", "promotion_valid_from"]
].copy()

issues["row_no"] = issues.index + 2

issues[["row_no", "pdp_url", "price_valid_from", "promotion_valid_from"]]

,row_no,pdp_url,price_valid_from,promotion_valid_from


In [45]:
import re

col = "regular_price"

# More than one decimal point in the numeric part
invalid = df[
    df[col]
    .fillna("")
    .astype(str)
    .str.contains(r'\d+\.\d+\.', regex=True)
]

print(f"Rows with multiple decimal points: {len(invalid)}")
print(invalid[[col]])

Rows with multiple decimal points: 0
Empty DataFrame
Columns: [regular_price]
Index: []


In [46]:
import re

url_pattern = re.compile(
    r"^https?://[^\s/$.?#].[^\s]*$",
    re.IGNORECASE
)

invalid_urls = df[
    
    df["pdp_url"].isna() |
    ~df["pdp_url"].astype(str).str.match(url_pattern)
]

print(f"Invalid URLs: {len(invalid_urls)}")
print(invalid_urls[[ "pdp_url"]])

Invalid URLs: 0
Empty DataFrame
Columns: [pdp_url]
Index: []


In [47]:
df.loc[df["pdp_url"].str.contains(r"\s", regex=True, na=False),
       ["pdp_url"]]

,pdp_url


In [48]:
import pandas as pd

# Promotion description has a value containing %
promo_mask = (
    df["promotion_description"].fillna("").astype(str).str.contains("%", na=False)
)

# Percentage discount is empty
discount_empty = (
    df["percentage_discount"].isna() |
    df["percentage_discount"].astype(str).str.strip().eq("")
)

# Both conditions
mask = promo_mask & discount_empty


issues = df.loc[
    mask,
    ["promotion_description", "percentage_discount", "pdp_url"]
].copy()

# Row number (assuming header is row 1)
issues["row_no"] = issues.index + 2

# Arrange columns
issues = issues[
    ["row_no", "pdp_url", "promotion_description", "percentage_discount"]
]

print(issues.to_string(index=False))

Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, percentage_discount]
Index: []


In [49]:
df.grammage_quantity.value_counts()

grammage_quantity
1        1061
500       646
4         506
200       484
400       446
75        442
250       391
6         298
150       298
100       283
2         235
300       219
750       181
450       169
20        159
3         150
350       146
12        142
70        141
50        134
8         132
125       130
10        128
5         115
90        113
120       102
80        102
800        94
30         94
40         94
330        89
60         78
160        76
180        72
340        67
190        57
45         55
1.5        55
280        53
600        53
130        52
170        51
140        50
25         50
320        48
85         47
454        47
220        46
227        46
360        45
440        43
15         43
110        41
18         40
135        39
380        37
275        37
24         37
240        36
375        35
270        31
48         31
35         30
700        29
16         29
900        29
420        29
9          28
425        28
210        27
16

In [50]:
mask = (
    df["promotion_valid_from"].notna()
    & df["promotion_valid_from"].astype(str).str.strip().ne("")
    & (
        df["price_valid_from"].isna()
        | df["price_valid_from"].astype(str).str.strip().eq("")
    )
)

issues = df.loc[
    mask,
    
    [ "pdp_url", "promotion_valid_from", "promotion_description", "price_valid_from"]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    ["row_no",  "pdp_url", "promotion_valid_from","promotion_description", "price_valid_from"]
]

print(issues.to_string(index=False))

Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_from, promotion_description, price_valid_from]
Index: []


In [51]:
df.price_valid_from.value_counts()

price_valid_from
05/11/2026    3234
15/10/2026      82
03/09/2026      59
29/09/2026      24
24/09/2026      23
17/09/2026      19
22/10/2026      18
06/11/2025      15
02/07/2026      15
11/06/2026      15
08/01/2026      14
01/05/2025      14
10/01/2025       9
23/03/2026       7
01/03/2023       7
06/03/2025       5
01/01/2026       5
15/08/2026       4
24/07/2025       4
14/11/2025       4
23/08/2024       4
13/07/2025       3
18/08/2026       3
30/04/2026       3
04/01/2024       3
06/06/2024       2
21/07/2022       2
26/09/2026       2
06/11/2026       2
23/03/2023       2
16/06/2022       2
12/09/2024       2
04/07/2026       2
18/02/2022       1
01/02/2024       1
23/09/2024       1
Name: count, dtype: int64

In [52]:
invalid_urls = invalid_urls["pdp_url"].unique()

print("Unique invalid URLs:", len(invalid_urls))
for url in invalid_urls:
    print(repr(url))

Unique invalid URLs: 0


In [53]:
import numpy as np

invalid_urls = np.unique(invalid_urls)

print("Unique invalid URLs:", len(invalid_urls))
for url in invalid_urls:
    print(url)  # Or your specific loop logic


Unique invalid URLs: 0


In [54]:
issues = df[
    df["product_name"].astype(str).str.startswith("Denner", na=False)
    & (df["brand"].fillna("").str.strip() != "Denner")
]

issues[["product_name", "brand", "pdp_url"]]

,product_name,brand,pdp_url


In [55]:
df.instock.value_counts()

instock
True     11219
False       66
Name: count, dtype: int64

In [56]:
import re

image_cols = [c for c in df.columns if c.startswith("image_url_")]

# Basic image URL pattern
pattern = re.compile(
    r"^https?://[^\s]+\.(jpg|jpeg|png|webp|gif|bmp|avif)(\?.*)?$",
    re.IGNORECASE
)

for col in image_cols:
    invalid = df[
        
        df[col].notna() &
        (df[col].astype(str).str.strip() != "") &
        ~df[col].astype(str).str.match(pattern)
    ]

    print(f"\n{col}: {len(invalid)} invalid URLs")

    if not invalid.empty:
        print(invalid[["unique_id", col]].head())


image_url_1: 11284 invalid URLs
    unique_id                                               image_url_1
0  1179723000  https://images.cdn.shop.supervalu.ie/detail/1179723000_1
1  1013680000  https://images.cdn.shop.supervalu.ie/detail/1013680000_1
2  1030326000    https://images.cdn.shop.supervalu.ie/zoom/1030326000_1
3  1085596000    https://images.cdn.shop.supervalu.ie/zoom/1085596000_1
4  1879149000    https://images.cdn.shop.supervalu.ie/zoom/1879149000_1

image_url_2: 6594 invalid URLs
    unique_id  \
0  1179723000   
1  1013680000   
2  1030326000   
5  1700369000   
6  1802422000   

                                                                                                                 image_url_2  
0  https://storage.googleapis.com/images-mus-prd-7e2fdbe.mus.prd.v8.commerce.mi9cloud.com/product-images/detail/1179723000_9  
1  https://storage.googleapis.com/images-mus-prd-7e2fdbe.mus.prd.v8.commerce.mi9cloud.com/product-images/detail/1013680000_9  
2    https://storag

In [57]:
(
    df.groupby(['store_addressline2', 'unique_id'])
      .size()
      .reset_index(name='count')
      .query('count > 1')
      .sort_values(['store_addressline2', 'count'], ascending=[True, False])
)

,store_addressline2,unique_id,count


In [58]:
import re

pattern = re.compile(r"^https?://[^\s]+$", re.IGNORECASE)

for col in image_cols:
    invalid = df[
        df[col].notna() &
        (df[col].astype(str).str.strip() != "") &
        ~df[col].astype(str).str.match(pattern)
    ]

    print(f"{col}: {len(invalid)} invalid URLs")

image_url_1: 0 invalid URLs
image_url_2: 0 invalid URLs
image_url_3: 0 invalid URLs
image_url_4: 0 invalid URLs
image_url_5: 0 invalid URLs
image_url_6: 0 invalid URLs


In [59]:
price_cols = [
    "regular_price",
    "selling_price",
   
   
   
]

# Keep only columns that exist in the file
price_cols = [c for c in price_cols if c in df.columns]

# Rows where at least one price column is empty
empty_price_rows = df[df[price_cols].replace("", pd.NA).isna().any(axis=1)]

print(f"Rows with empty price fields: {len(empty_price_rows)}")
display(empty_price_rows[["unique_id","pdp_url", "promotion_description","product_name"] + price_cols])

Rows with empty price fields: 0


,unique_id,pdp_url,promotion_description,product_name,regular_price,selling_price


In [60]:
import pandas as pd

# Treat empty strings as missing values
promo_price = df["promotion_price"].replace("", pd.NA)
promo_desc = df["promotion_description"].replace("", pd.NA)

# 1. Promotion price present but promotion description missing
price_no_desc = df[
    promo_price.notna() & promo_desc.isna()
]

# 2. Promotion description present but promotion price missing
desc_no_price = df[
    promo_desc.notna() & promo_price.isna()
]

print("Promotion price present but description missing:", len(price_no_desc))
display(price_no_desc[["unique_id", "product_name", "promotion_price","pdp_url", "promotion_description"]])

print("\nPromotion description present but price missing:", len(desc_no_price))
display(desc_no_price[["unique_id", "product_name", "promotion_price", "pdp_url","promotion_description"]])

Promotion price present but description missing: 0


,unique_id,product_name,promotion_price,pdp_url,promotion_description



Promotion description present but price missing: 1043


,unique_id,product_name,promotion_price,pdp_url,promotion_description
9,1683185003,SuperValu Signature Tastes Blueberries (225 g),NaN,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-blueberries-225-g-id-1683185003,3 for €10
11,1386984000,Keelings Strawberries Family Pack (350 g),NaN,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/keelings-strawberries-family-pack-350-g-id-1386984000,3 for €10
21,1998084000,SuperValu XL Mangos Ready To Eat (2 Piece),NaN,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-xl-mangos-ready-to-eat-2-piece-id-1998084000,3 for €10
25,1892011000,SuperValu Signature Tastes Kiwi Gold Tray (5 Piece),NaN,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-kiwi-gold-tray-5-piece-id-1892011000,3 for €10
28,1917042000,SuperValu Signature Tastes Mixed Berries (200 g),NaN,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-mixed-berries-200-g-id-1917042000,3 for €10
37,1892085001,SuperValu Strawberries (350 g),NaN,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-strawberries-350-g-id-1892085001,3 for €10
43,1628438002,SuperValu Signature Tastes Red Grapes (500 g),NaN,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-red-grapes-500-g-id-1628438002,3 for €10
44,1626884000,SuperValu Signature Tastes Green Grapes (500 g),NaN,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-green-grapes-500-g-id-1626884000,3 for €10
51,1628438001,SuperValu Signature Tastes Black Grapes (500 g),NaN,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-black-grapes-500-g-id-1628438001,3 for €10
60,1434795000,SuperValu Large Mixed Grapes (650 g),NaN,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-large-mixed-grapes-650-g-id-1434795000,3 for €10


In [61]:
df.breadcrumb.value_counts()

breadcrumb
Food Cupboard > Chocolate & Sweets > Chocolate Bars                                                  284
Food Cupboard > Crisps, Snacks, Nuts & Popcorn > Crisps & Snacks                                     187
Food Cupboard > Chocolate & Sweets > Sweets                                                          161
Food Cupboard > Packet Sauces, Stocks & Herbs > Herbs & Spices                                       119
Wine, Beer & Spirits > Beer > Lager                                                                  113
Food Cupboard > Biscuits > Everyday Biscuits                                                         111
Drinks > Soft Drinks > Sports & Energy Drinks                                                        101
Food Cupboard > Foods Of The World > Italian > Italian Sauces                                         93
Milk, Yogurt, Butter & Eggs > Yogurts > Protein                                                       84
Meat & Poultry > Chicken & Poultry > Pre-pac

In [62]:
invalid = df[df["breadcrumb"].str.contains("…", na=False)]

print(f"Rows with truncated breadcrumbs: {len(invalid)}")
display(invalid[["unique_id", "breadcrumb"]])

Rows with truncated breadcrumbs: 0


,unique_id,breadcrumb


In [63]:
df.competitor_name.value_counts()

competitor_name
supervalu    11285
Name: count, dtype: int64

In [64]:
df.variants.value_counts()

Series([], Name: count, dtype: int64)

In [65]:
result = df[
    df["selling_price"].isna() & df["regular_price"].isna()

text_cols = df.select_dtypes(include="object").columns

issues = {}

for col in text_cols:
    # Match | not preceded by a backslash
    mask = (
        df[col]
        .fillna("")
        .astype(str)
        .str.contains(r"(?<!\\)\|", regex=True)
    )

    if mask.any():
        issues[col] = df.loc[mask, ["unique_id", col]]

# Print results
if issues:
    print("Columns containing unescaped '|':\n")
    for col, data in issues.items():
        print(f"\n=== {col} ({len(data)} rows) ===")
        print(data.to_string(index=False))
else:
    print("No unescaped '|' found.")
]

print(result)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3466212421.py, line 2)

In [66]:
import pandas as pd

# Text columns only
text_cols = df.select_dtypes(include="object").columns

invalid_chars = [ "\n", "\r", "\t"]

issues = {}

for col in text_cols:
    mask = df[col].fillna("").astype(str).str.contains(r"[\n\r\t]", regex=True)
    if mask.any():
        issues[col] = df.loc[mask, ["unique_id", col]]

# Summary
if issues:
    print("Columns containing \\n, \\r or \\t:\n")
    for col, data in issues.items():
        print(f"{col}: {len(data)} rows")
else:
    print("No \\n, \\r or \\t found.")
    

No \n, \r or \t found.


In [67]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

results = []

for col in text_cols:
    mask = df[col].fillna("").astype(str).str.contains(r"[\n\r\t]", regex=True)

    for idx, value in df.loc[mask, col].items():
        results.append({
            "row_no": idx + 2,      # Excel row number
            "unique_id": df.at[idx, "unique_id"],
            "pdp_url": df.at[idx, "pdp_url"],
            "column": col,
            "issue_value": value
        })

issues_df = pd.DataFrame(results)

issues_df

""


In [68]:
columns = ["product_description", "special_information", "manufacturer_address"]

for col in columns:
    rows = df[
        df[col]
        .fillna("")
        .astype(str)
        .str.contains("|", regex=False, na=False)
    ]

    print(f"\n=== {col}: {len(rows)} rows ===")

    if not rows.empty:
        display(rows[["unique_id", "pdp_url", col]])


=== product_description: 13 rows ===


,unique_id,pdp_url,product_description
1146,1783485000,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/naked-unsmoked-back-bacon-200-g-id-1783485000,"For more than a decade, I insisted we not touch Bacon until we could make it without nitrites. In 2017, we proudly pioneered Naked Bacon - made without harmful nitrites while still naturally tasty.We saved your Bacon!Denis Lynn \| founder"
6056,1623310004,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/max-strong-hot-chicken-wing-140-g-id-1623310004,"-Walkers Max Strong Hot Chicken Wings Crisps -Irresistible, hot and spicy deep-ridged crisps for a bolder taste experience -This sharing bag is the ideal snack for a party, to share with your mates or great for a big night in -No artificial colours and no MSG -Suitable for vegetarians\|-Walkers Max Strong Hot Chicken Wings Crisps-Irresistible, hot and spicy deep-ridged crisps for a bolder taste experience-This sharing bag is the ideal snack for a party, to share with your mates or great for a big night in-No artificial colours and no MSG-Suitable for vegetarians"
8823,1734572000,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/poligrip-max-seal-denture-fixative-40-g-id-1734572000,"Feel the freedom of relishing your favourite foods. Poligrip Max Hold and Seal gives you strongest hold* and forms an all-around seal to provide a 5x protective food barrier**. It provides an up to 12-hour hold, and the precision nozzle helps you to apply exactly the right amount. Enjoy the freedom from foods such as nuts, seeds or popcorn sneaking under your partial or full denture and savour every bite. The adhesive also helps cushion gums and works all day to help protect gums. Plus, it's free of zinc and contains no added colours or flavours. So, it won't get in the way of fully appreciating the taste of your food. Enjoy life worry-free. Applying the adhesive is easy - simply clean and dry your denture and rinse your mouth. Then refer to pack diagrams for how to apply the product to either partial or full dentures. Next, press your denture into place, holding it firmly. Bite down for a few seconds, and you're good to go. To remove, swish your mouth with water first and then use a rocking motion. Boost your partial or full denture care routine and clean with Poligrip cleaning tablets. They kill 10x more bacteria than regular toothpaste***. Give them a try to help keep your denture clean, fresh and hygienic. *Within Poligrip range **Protects against 5X more food particles vs no adhesive ***In laboratory testing. Soaking only 1. MAX HOLD: Strongest hold* for your dentures - suitable for partial and full dentures. Just eat what you really love! *Within Poligrip range2. SEAL: Forms an all-round seal to provide 5x protective food barrier** against food particles. **Protects against 5X more food particles vs no adhesive3. PRECISE APPLICATION: Precision nozzle for even application helps you to apply exactly the right amount.4. FEAR NO FOOD: Provides 12-hour hold. You can go for what you really crave.5. ZINC FREE: And does not contain artificial colours and flavours.6. WORLD'S NO.1 DENTURE CARE BRAND±: Boost your routine with Poligrip denture cleansers. They kill 10x more bacteria than regular toothpaste.± ±Value Retail Sales data for GSK-branded denture care products, 12 months ending Sept 2022 ±Soaking only in lab tests.\|ITEM DETAILS: One tube of 40g flavour free and neutral-coloured denture adhesive with precision nozzle. Carton is made with recycled material and is recyclable."
9078,1866776002,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/aveeno-kids-shampoo-250-ml-id-1866776002,"- For kids' developing skin- For sensitive skin & scalp- With soothing oat & sheer butter- Gently cleanses leaving different hair types easy to manage- Tear-freeNourishing care for every journey through the deep dark wood with AVEENO® Kids Shampoo 250ml Gruffalo Edition!* \| Enriched with Soothing Oat & Shea Butter \| Children's Shampoo Developed fo


=== special_information: 0 rows ===

=== manufacturer_address: 0 rows ===


In [69]:
import pandas as pd
import re




df["dimensions"] = df["dimensions"].fillna("").str.strip()

pattern = re.compile(
    r'^\d+(\.\d+)?(\s*[a-zA-Z]+)?\s*[xX×]\s*'
    r'\d+(\.\d+)?(\s*[a-zA-Z]+)?\s*[xX×]\s*'
    r'\d+(\.\d+)?(\s*[a-zA-Z]+)?$'
)

invalid = df[
    (df["dimensions"] != "") &
    (~df["dimensions"].str.match(pattern))
]

print(f"Invalid values: {len(invalid)}")
print(invalid[["unique_id", "product_name", "dimensions"]])

Invalid values: 0
Empty DataFrame
Columns: [unique_id, product_name, dimensions]
Index: []


In [70]:
import pandas as pd

issues = []

for col in df.columns:
    if df[col].dtype == "object":
        mask = df[col].fillna("").astype(str).str.contains(
            r'^\s+|\s+$|\t|\n|\r| {2,}',
            regex=True,
            na=False
        )

        if mask.any():
            temp = df.loc[mask, [col]].copy()
            temp["row_no"] = temp.index + 2
            temp["column"] = col
            issues.append(temp[["row_no", "column", col]])

if issues:
    result = pd.concat(issues, ignore_index=True)
    print(result)
else:
    print("No unwanted whitespace found.")

    row_no                   column  \
0      791  nutritional_information   
1     3301  nutritional_information   
2     3394  nutritional_information   
3     4440  nutritional_information   
4     4488  nutritional_information   
5     4862  nutritional_information   
6     4898  nutritional_information   
7     6200  nutritional_information   
8     6465  nutritional_information   
9     6886  nutritional_information   
10    7059  nutritional_information   
11    7222  nutritional_information   

                                                                                                                                                                                                                                                                                                                                                                                                                                          nutritional_information  
0                     {' Grilled Per 100

In [71]:

# Replace empty strings with NA
df = df.replace(r'^\s*$', pd.NA, regex=True)

# Filter Fleisch & Fisch
ff = df[df["producthierarchy_level1"] == "Fleisch & Fisch"]

# Subcategories that should have price_per_unit
ppu_subcats = ["Fleisch & Geflügel", "Frischfisch & Schalentiere"]

# Missing price_per_unit
invalid_ppu = ff[
    ff["producthierarchy_level2"].isin(ppu_subcats) &
    ff["price_per_unit"].isna()
]

print("Missing price_per_unit:")
print(invalid_ppu[["unique_id","product_name","producthierarchy_level2","price_per_unit"]])

Missing price_per_unit:
Empty DataFrame
Columns: [unique_id, product_name, producthierarchy_level2, price_per_unit]
Index: []


In [72]:
date_columns = [
    "promotion_valid_upto",
    "promotion_valid_from",
    "price_valid_from"
]

for col in date_columns:
    print("\nColumn:", col)
    print(df[col].value_counts(dropna=False))


Column: promotion_valid_upto
promotion_valid_upto
NaN           7673
19/11/2026    2420
12/11/2026     947
01/01/2050     189
18/11/2026      40
11/11/2026      11
09/11/2026       4
08/11/2026       1
Name: count, dtype: int64

Column: promotion_valid_from
promotion_valid_from
NaN           7673
05/11/2026    3234
15/10/2026      82
03/09/2026      59
29/09/2026      24
24/09/2026      23
17/09/2026      19
22/10/2026      18
06/11/2025      15
02/07/2026      15
11/06/2026      15
08/01/2026      14
01/05/2025      14
10/01/2025       9
23/03/2026       7
01/03/2023       7
06/03/2025       5
01/01/2026       5
15/08/2026       4
24/07/2025       4
14/11/2025       4
23/08/2024       4
13/07/2025       3
18/08/2026       3
30/04/2026       3
04/01/2024       3
06/06/2024       2
21/07/2022       2
26/09/2026       2
06/11/2026       2
23/03/2023       2
16/06/2022       2
12/09/2024       2
04/07/2026       2
18/02/2022       1
01/02/2024       1
23/09/2024       1
Name: count, dtyp

In [73]:
df.currency.value_counts(dropna=False)

currency
EUR    11285
Name: count, dtype: int64

In [74]:
invalid_discount = df[
    df["percentage_discount"]
    .astype(str)
    .str.contains(r"[%\-]", regex=True, na=False)
]

print("Invalid percentage_discount values:")
print(invalid_discount[["unique_id", "product_name", "percentage_discount"]])

print("\nCount:", len(invalid_discount))

Invalid percentage_discount values:
Empty DataFrame
Columns: [unique_id, product_name, percentage_discount]
Index: []

Count: 0


In [75]:
s = df['percentage_discount']

mask = (
    s.notna() &
    s.astype(str).str.strip().ne("") &
    ~s.astype(str).str.fullmatch(r'\d+', na=False)
)

issues = df.loc[
    mask,
    ['pdp_url', 'percentage_discount']
].copy()

issues['row_no'] = issues.index + 2

issues = issues[['row_no', 'pdp_url', 'percentage_discount']]

print(issues.to_string(index=False))
print(f"\nTotal issues: {len(issues)}")

 row_no                                                                                                                                                      pdp_url  percentage_discount
    776                                        https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-fresh-irish-beef-striploin-steak-1-kg-id-1023229001                 33.0
    921                                          https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-fresh-irish-off-leg-lamb-prima-1-kg-id-1344136000                 33.0
    929                                                    https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-fresh-irish-lamb-leg-1-kg-id-1023336001                 33.0
   1227                                                                https://shop.supervalu.ie/sm/delivery/rsid/5550/product/fresh-salmon-joint-1-kg-id-1940520001                 25.0
   1373                                                              h

In [76]:
invalid = df[
    df["promotion_description"].notna() &
    (
        df["promotion_valid_from"].isna() |
        df["promotion_valid_upto"].isna()
    )
]

print(f"Rows with promotion description but missing promotion dates: {len(invalid)}")

display(
    invalid[
        [
            "promotion_price",
            "promotion_description",
            "promotion_valid_from",
            "promotion_valid_upto",
            "pdp_url"
        ]
    ]
)

Rows with promotion description but missing promotion dates: 0


,promotion_price,promotion_description,promotion_valid_from,promotion_valid_upto,pdp_url


In [77]:
mask = (
    df["regular_price"].notna() &
    df["price_was"].notna() &
    (pd.to_numeric(df["regular_price"], errors="coerce") !=
     pd.to_numeric(df["price_was"], errors="coerce"))
)

issues = df.loc[mask, ["pdp_url", "regular_price", "price_was"]].copy()
issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "regular_price", "price_was"]]

issues

,row_no,pdp_url,regular_price,price_was


In [78]:
mask = (
    df["unique_id"].astype(str).fillna("") + "P"
    !=
    df["product_unique_key"].astype(str).fillna("")
)

issues = df.loc[mask, ["unique_id", "product_unique_key", "pdp_url"]].copy()
issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "unique_id", "product_unique_key"]]

issues

,row_no,pdp_url,unique_id,product_unique_key


In [79]:
import pandas as pd

# Normalize for case-insensitive comparison
df["site_shown_uom"] = df["site_shown_uom"].fillna("").astype(str)
df["grammage_unit"] = df["grammage_unit"].fillna("").astype(str)
df["grammage_quantity"] = df["grammage_quantity"].fillna("").astype(str)

# -----------------------------
# Rule 1: Teebeutel / Beutel -> btl
# -----------------------------
btl_keywords = [
    "Teebeutel Packung",
    "Teebeutel",
    "Beutel",
    "Btl",
    "Teebeutel Karton",
    "Teebeutel Paket",
    "Packung Beutel"
]

btl_invalid = df[
    df["site_shown_uom"].str.contains("|".join(btl_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "btl")
]

print(f"BTL unit mismatches: {len(btl_invalid)}")
display(btl_invalid[[
    "pdp_url",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit"
]])

# -----------------------------
# Rule 2: Portion Packung / ANW -> stuck
# -----------------------------
stuck_keywords = [
    "Portion Packung",
    "ANW"
]

stuck_invalid = df[
    df["site_shown_uom"].str.contains("|".join(stuck_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "stuck")
]

print(f"STUCK unit mismatches: {len(stuck_invalid)}")
display(stuck_invalid[[
    "unique_id",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit"
]])

# -----------------------------
# Rule 3: wg / Waschgänge -> wg
# -----------------------------
wg_keywords = [
    "wg",
    "Waschgänge"
]

wg_invalid = df[
    df["site_shown_uom"].str.contains("|".join(wg_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "wg")
]

print(f"WG unit mismatches: {len(wg_invalid)}")
display(wg_invalid[[
    "unique_id",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit",
    "pdp_url"
]])

BTL unit mismatches: 1


,pdp_url,product_name,site_shown_uom,grammage_quantity,grammage_unit
11092,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/camel-subtle-blue-cigarettes-20-piece-id-1019009000,Camel Subtle Blue Cigarettes (20 Piece),Camel Subtle Blue Cigarettes (20 Piece),20,pc


STUCK unit mismatches: 0


,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit


WG unit mismatches: 2


,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit,pdp_url
4425,1282089000,Newgrange Gold Garlic & Mild Chilli Irish Rapeseed Oil (250 ml),Newgrange Gold Garlic & Mild Chilli Irish Rapeseed Oil (250 ml),250,ml,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/newgrange-gold-garlic-mild-chilli-irish-rapeseed-oil-250-ml-id-1282089000
4434,1282087000,Newgrange Gold Garden Herb & Lemon Irish Rapeseed Oil (250 ml),Newgrange Gold Garden Herb & Lemon Irish Rapeseed Oil (250 ml),250,ml,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/newgrange-gold-garden-herb-lemon-irish-rapeseed-oil-250-ml-id-1282087000


In [80]:
import pandas as pd

price_cols = ["regular_price", "selling_price"]

for col in price_cols:
    # Ignore missing values
    mask = (
        df[col].notna() &
        (~df[col].astype(float).round(2).eq(df[col].astype(float)))
    )

    invalid = df.loc[mask, ["unique_id", "product_name", col]]

    print(f"\n{col}: {len(invalid)} values not in 2-decimal format")
    display(invalid)


regular_price: 0 values not in 2-decimal format


,unique_id,product_name,regular_price



selling_price: 0 values not in 2-decimal format


,unique_id,product_name,selling_price


In [82]:
import pandas as pd

# Match both old and new site keywords
promo_mask = (
    df["promotion_description"]
    .fillna("")
    .astype(str)
    .str.contains(r"Trajno znižano|Znižano", case=False, na=False)
)

# Empty promotion price
promo_price_empty = (
    df["promotion_price"].isna()
    | df["promotion_price"].astype(str).str.strip().eq("")
)

# Check where promotion price is NOT empty
promo_price_issue = promo_mask & ~promo_price_empty

# Get all matching records
issues = df.loc[
    promo_mask,
    [
        "unique_id",
        "pdp_url",
        "promotion_description",
        "regular_price",
        "selling_price",
        "promotion_price"
    ]
].copy()

issues["row_no"] = issues.index + 2

# Add issue remarks
def check_issue(row):
    remarks = []

    if pd.notna(row["promotion_price"]) and str(row["promotion_price"]).strip() != "":
        remarks.append("promotion_price should be empty")

    if row["regular_price"] != row["selling_price"]:
        remarks.append("regular_price and selling_price are not equal")

    return "; ".join(remarks)

issues["issue"] = issues.apply(check_issue, axis=1)

# Show only violations
issues = issues[issues["issue"] != ""]

issues = issues[
    [
        "row_no",
        "unique_id",
        "pdp_url",
        "promotion_description",
        "regular_price",
        "selling_price",
        "promotion_price",
        "issue"
    ]
]

print(issues.to_string(index=False))

Empty DataFrame
Columns: [row_no, unique_id, pdp_url, promotion_description, regular_price, selling_price, promotion_price, issue]
Index: []


In [83]:
import re
import pandas as pd

# Normalize grammage_unit
df["grammage_unit_norm"] = df["grammage_unit"].fillna("").str.lower().str.strip()

# Extract quantity from site_shown_uom
df["site_qty"] = (
    df["site_shown_uom"]
      .str.extract(r"(\d+(?:[.,]\d+)?)", expand=False)
      .str.replace(",", ".", regex=False)
)

df["site_qty"] = pd.to_numeric(df["site_qty"], errors="coerce")

# ---------- Rule 1 : Beutel / Btl ----------
btl_pattern = (
    r"Teebeutel Packung|Teebeutel Karton|Teebeutel Paket|"
    r"Packung Beutel|Teebeutel|Beutel|Btl"
)

btl_issue = df[
    df["site_shown_uom"].fillna("").str.contains(btl_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "btl") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

# ---------- Rule 2 : Portion Packung / ANW ----------
stuck_pattern = r"Portion Packung|ANW"

stuck_issue = df[
    df["site_shown_uom"].fillna("").str.contains(stuck_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "stuck") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

# ---------- Rule 3 : Waschgang / Waschgänge / wg ----------
wg_pattern = r"\bwg\b|Waschgänge|Waschgang"

wg_issue = df[
    df["site_shown_uom"].fillna("").str.contains(wg_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "wg") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

print("Btl issues:", len(btl_issue))
print("Stuck issues:", len(stuck_issue))
print("WG issues:", len(wg_issue))

Btl issues: 1
Stuck issues: 0
WG issues: 0


In [84]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

results = []

for col in text_cols:
    mask = df[col].fillna("").astype(str).str.contains(r" {2,}", regex=True)

    for idx, value in df.loc[mask, col].items():
        results.append({
            "row_no": idx + 2,
            "pdp_url": df.at[idx, "pdp_url"],
            "column": col,
            "issue_value": repr(value)
        })

issues_df = pd.DataFrame(results)
issues_df

,row_no,pdp_url,column,issue_value
0,791,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-fresh-irish-gourmet-sirloin-beef-burgers-2-pack-260-g-id-1574099000,nutritional_information,"""{' Grilled Per 100g': {'Energy': '687.0kJ', 'Calories': '163.0kcal', 'Total Fat': '5.5g', 'Saturates': '2.4g', 'Carbohydrate': '4.4g', 'Sugars': '0.5g', 'Fibre': '0.5g', 'Protein': '24.0g', 'Sodium': '0.82g'}, ' Grilled Per 104g serving': {'Energy': '715.0kJ', 'Calories': '170.0kcal', 'Total Fat': '5.7g', 'Saturates': '2.5g', 'Carbohydrate': '4.6g', 'Sugars': '0.5g', 'Fibre': '0.5g', 'Protein': '25.0g', 'Sodium': '0.85g'}}"""
1,3301,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/clean-cut-meals-chicken-dahl-protein-pot-350-g-id-2026768000,nutritional_information,"""{'(As microwaved): per 100g': {'Energy': '454.0kJ', 'Calories': '109.0kcal', 'Total Fat': '5.5g', 'Saturates': '4.1g', 'Carbohydrate': '5.8g', 'Sugars': '1.9g', 'Fibre': '5.0g', 'Protein': '6.6g', 'Sodium': '0.9g'}, '(As microwaved): Per 350g Serving': {'Energy': '1578.0kJ', 'Calories': '379.0kcal', 'Total Fat': '19.0g', 'Saturates': '14.0g', 'Carbohydrate': '20.0g', 'Sugars': '6.7g', 'Fibre': '18.0g', 'Protein': '23.0g', 'Sodium': '3.2g'}}"""
2,3394,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/caramico-margherita-pizza-430-g-id-1956595001,nutritional_information,"""{'oven cooked per 100g': {'Energy': '954.0kJ', 'Calories': '227.0kcal', 'Total Fat': '7.2g', 'Saturates': '3.8g', 'Carbohydrate': '27.0g', 'Sugars': '2.6g', 'Fibre': '3.1g', 'Protein': '12.0g', 'Sodium': '0.91g'}, 'oven cooked per 1/2 pizza (198g)': {'Energy': '1876.0kJ', 'Calories': '446.0kcal', 'Total Fat': '14.0g', 'Saturates': '7.5g', 'Carbohydrate': '53.0g', 'Sugars': '5.1g', 'Fibre': '6.1g', 'Protein': '24.0g', 'Sodium': '1.8g'}}"""
3,4440,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/walkers-wotsits-crunchy-sweet-spicy-flamin-hot-corn-puffs-140-g-id-1815378001,nutritional_information,"""{'per 100g': {'Energy': '2280.0kJ', 'Calories': '546.0kcal', 'Total Fat': '33.0g', 'Saturates': '4.2g', 'Carbohydrate': '56.0g', 'Sugars': '3.3g', 'Fibre': '1.0g', 'Protein': '5.3g', 'Sodium': '1.5g'}, 'per 30g serving:': {'Energy': '684.0kJ', 'Calories': '164.0kcal', 'Total Fat': '9.9g', 'Saturates': '1.3g', 'Carbohydrate': '17.0g', 'Sugars': '1.0g', 'Fibre': '0.3g', 'Protein': '1.6g', 'Sodium': '0.45g'}}"""
4,4488,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/blue-dragon-wholewheat-4-nests-noodles-250-g-id-1916259001,nutritional_information,"""{'as prepared Per 100g': {'Energy': '698.0kJ', 'Calories': '165.0kcal', 'Total Fat': '0.7g', 'Saturates': '0.1g', 'Carbohydrate': '30.0g', 'Sugars': '0.8g', 'Fibre': '4.8g', 'Protein': '7.0g', 'Sodium': '0.22g'}, 'as prepared 144g (approx.) Per Serving': {'Energy': '1005.0kJ', 'Calories': '238.0kcal', 'Total Fat': '1.0g', 'Saturates': '0.1g', 'Carbohydrate': '44.0g', 'Sugars': '1.2g', 'Fibre': '6.9g', 'Protein': '10.0g', 'Sodium': '0.31g'}}"""
5,4862,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/walkers-wotsits-crunchy-really-cheesy-corn-puffs-140-g-id-1815378000,nutritional_information,"""{'per 100g': {'Energy': '2304.0kJ', 'Calories': '552.0kcal', 'Total Fat': '34.0g', 'Saturates': '4.8g', 'Carbohydrate': '56.0g', 'Sugars': '5.8g', 'Fibre': '0.9g', 'Protein': '5.1g', 'Sodium': '1.3g'}, 'per 30g serving:': {'Energy': '691.0kJ', 'Calories': '166.0kcal', 'Total Fat': '10.0g', 'Saturates': '1.4g', 'Carbohydrate': '17.0g', 'Sugars': '1.7g', 'Fibre': '0.3g', 'Protein': '1.5g', 'Sodium': '0.39g'}}"""
6,4898,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/quavers-cheese-34-g-id-1276543000,nutritional_information,"""{'per 100g': {'Energy': '2245.0kJ', 'Calories': '536.0kcal', 'Total Fat': '30.8g', 'Saturates': '2.7g', 'Carbohydrate': '62.1g', 'Sugars': '2.7g', 'Fibre': '1.2g', 'Protein': '2.5g', 'Sodium': '2.14g'}, 'per 34g Pack': {'Energy': '674.0kJ', 'Calories': '161.0kcal', 'Total Fat': '9.2g', 'Saturates': '0.8g', 'Carboh

In [85]:
btl_issue[
    [
        "unique_id",
        "product_name",
        "site_shown_uom",
        "grammage_quantity",
        "grammage_unit",
        "site_qty"
    ]
].head(20)

,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit,site_qty
11092,1019009000,Camel Subtle Blue Cigarettes (20 Piece),Camel Subtle Blue Cigarettes (20 Piece),20,pc,20.0


In [86]:
wrong_unit_wg = wg_issue[
    wg_issue["grammage_unit_norm"] != "wg"
]

wrong_qty_wg = wg_issue[
    (wg_issue["grammage_unit_norm"] == "wg") &
    (wg_issue["grammage_quantity"] != wg_issue["site_qty"])
]

print("WG - Wrong unit:", len(wrong_unit_wg))
print("WG - Wrong quantity:", len(wrong_qty_wg))

WG - Wrong unit: 0
WG - Wrong quantity: 0


In [87]:
wrong_unit_wg[
    [
        "unique_id",
        "product_name",
        "site_shown_uom",
        "grammage_quantity",
        "grammage_unit",
        "pdp_url"
    ]
]

,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit,pdp_url


In [88]:
# JO PREIS price structure
jo_mask = df['promotion_description'].fillna('').astype(str).str.contains(
    r'JO PREIS',
    case=False,
    na=False
)

issues = df.loc[
    jo_mask & (
        df['regular_price'].isna() |
        df['selling_price'].isna() |
        df['price_per_unit'].isna()
    ),
    [
        'pdp_url',
        'promotion_description',
        'regular_price',
        'selling_price',
        'price_per_unit'
    ]
].copy()

issues['row_no'] = issues.index + 2

issues = issues[
    ['row_no', 'pdp_url', 'promotion_description',
     'regular_price', 'selling_price', 'price_per_unit']
]

print(f"JO PREIS issues: {len(issues)}")
print(issues.to_string(index=False))

JO PREIS issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, regular_price, selling_price, price_per_unit]
Index: []


In [89]:
text_cols = df.select_dtypes(include="object").columns

leading_trailing = {}

for col in text_cols:
    mask = (
        df[col].fillna("").astype(str)
        != df[col].fillna("").astype(str).str.strip()
    )
    if mask.any():
        leading_trailing[col] = df.loc[mask, col]

print(leading_trailing.keys())

dict_keys([])


In [90]:
multiple_spaces = {}

for col in text_cols:
    mask = df[col].fillna("").str.contains(r"\s{2,}", regex=True)
    if mask.any():
        multiple_spaces[col] = df.loc[mask, col]

print(multiple_spaces.keys())

dict_keys(['nutritional_information'])


In [91]:
special_space = {}

for col in text_cols:
    mask = df[col].fillna("").str.contains(r"[\t\r\n]", regex=True)
    if mask.any():
        special_space[col] = df.loc[mask, col]

print(special_space.keys())

dict_keys([])


In [92]:
special_space

{}

In [93]:
issues = []

for col in text_cols:
    mask = (
        (df[col].fillna("") != df[col].fillna("").str.strip()) |
        df[col].fillna("").str.contains(r"\s{2,}", regex=True) |
        df[col].fillna("").str.contains(r"[\t\r\n]", regex=True) |
        df[col].fillna("").str.contains("\u00A0", regex=False)
    )

    if mask.any():
        tmp = df.loc[mask, ["unique_id", col]].copy()
        tmp["column"] = col
        issues.append(tmp)

issues = pd.concat(issues, ignore_index=True)

issues.head()

,unique_id,nutritional_information,column
0,1574099000,"{' Grilled Per 100g': {'Energy': '687.0kJ', 'Calories': '163.0kcal', 'Total Fat': '5.5g', 'Saturates': '2.4g', 'Carbohydrate': '4.4g', 'Sugars': '0.5g', 'Fibre': '0.5g', 'Protein': '24.0g', 'Sodium': '0.82g'}, ' Grilled Per 104g serving': {'Energy': '715.0kJ', 'Calories': '170.0kcal', 'Total Fat': '5.7g', 'Saturates': '2.5g', 'Carbohydrate': '4.6g', 'Sugars': '0.5g', 'Fibre': '0.5g', 'Protein': '25.0g', 'Sodium': '0.85g'}}",nutritional_information
1,2026768000,"{'(As microwaved): per 100g': {'Energy': '454.0kJ', 'Calories': '109.0kcal', 'Total Fat': '5.5g', 'Saturates': '4.1g', 'Carbohydrate': '5.8g', 'Sugars': '1.9g', 'Fibre': '5.0g', 'Protein': '6.6g', 'Sodium': '0.9g'}, '(As microwaved): Per 350g Serving': {'Energy': '1578.0kJ', 'Calories': '379.0kcal', 'Total Fat': '19.0g', 'Saturates': '14.0g', 'Carbohydrate': '20.0g', 'Sugars': '6.7g', 'Fibre': '18.0g', 'Protein': '23.0g', 'Sodium': '3.2g'}}",nutritional_information
2,1956595001,"{'oven cooked per 100g': {'Energy': '954.0kJ', 'Calories': '227.0kcal', 'Total Fat': '7.2g', 'Saturates': '3.8g', 'Carbohydrate': '27.0g', 'Sugars': '2.6g', 'Fibre': '3.1g', 'Protein': '12.0g', 'Sodium': '0.91g'}, 'oven cooked per 1/2 pizza (198g)': {'Energy': '1876.0kJ', 'Calories': '446.0kcal', 'Total Fat': '14.0g', 'Saturates': '7.5g', 'Carbohydrate': '53.0g', 'Sugars': '5.1g', 'Fibre': '6.1g', 'Protein': '24.0g', 'Sodium': '1.8g'}}",nutritional_information
3,1815378001,"{'per 100g': {'Energy': '2280.0kJ', 'Calories': '546.0kcal', 'Total Fat': '33.0g', 'Saturates': '4.2g', 'Carbohydrate': '56.0g', 'Sugars': '3.3g', 'Fibre': '1.0g', 'Protein': '5.3g', 'Sodium': '1.5g'}, 'per 30g serving:': {'Energy': '684.0kJ', 'Calories': '164.0kcal', 'Total Fat': '9.9g', 'Saturates': '1.3g', 'Carbohydrate': '17.0g', 'Sugars': '1.0g', 'Fibre': '0.3g', 'Protein': '1.6g', 'Sodium': '0.45g'}}",nutritional_information
4,1916259001,"{'as prepared Per 100g': {'Energy': '698.0kJ', 'Calories': '165.0kcal', 'Total Fat': '0.7g', 'Saturates': '0.1g', 'Carbohydrate': '30.0g', 'Sugars': '0.8g', 'Fibre': '4.8g', 'Protein': '7.0g', 'Sodium': '0.22g'}, 'as prepared 144g (approx.) Per Serving': {'Energy': '1005.0kJ', 'Calories': '238.0kcal', 'Total Fat': '1.0g', 'Saturates': '0.1g', 'Carbohydrate': '44.0g', 'Sugars': '1.2g', 'Fibre': '6.9g', 'Protein': '10.0g', 'Sodium': '0.31g'}}",nutritional_information


In [94]:
import pandas as pd

# Convert to datetime for proper date comparison
price_date = pd.to_datetime(df["price_valid_from"], errors="coerce")
promo_date = pd.to_datetime(df["promotion_valid_from"], errors="coerce")

# Find rows where both have values but are different
mask = (
    price_date.notna()
    & promo_date.notna()
    & (price_date != promo_date)
)

issues = df.loc[
    mask,
    ["unique_id", "pdp_url", "price_valid_from", "promotion_valid_from"]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    ["row_no", "unique_id", "pdp_url",
     "price_valid_from", "promotion_valid_from"]
]

print(issues.to_string(index=False))
print(f"\nTotal issues: {len(issues)}")

Empty DataFrame
Columns: [row_no, unique_id, pdp_url, price_valid_from, promotion_valid_from]
Index: []

Total issues: 0


/tmp/ipykernel_74904/158468668.py:4: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  price_date = pd.to_datetime(df["price_valid_from"], errors="coerce")
/tmp/ipykernel_74904/158468668.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  promo_date = pd.to_datetime(df["promotion_valid_from"], errors="coerce")


In [95]:
import pandas as pd

# Convert price columns to numeric
cols = ["regular_price", "selling_price", "promotion_price", "price_per_unit"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

invalid = df[
    # Mehrweg product
    (
        df["product_name"].fillna("").str.contains("Mehrweg", case=False) |
        df["breadcrumb"].fillna("").str.contains("Mehrweg", case=False)
    )
    &
    # Has AKTION promotion
    df["promotion_description"].fillna("").str.contains("AKTION", case=False)
    &
    # Rule violation
    (
        df["regular_price"].notna() |
        (df["selling_price"] != df["price_per_unit"]) |
        (df["promotion_price"] != df["price_per_unit"])
    )
]

print("Invalid rows:", len(invalid))

invalid[
    [
        "unique_id",
        "product_name",
        "regular_price",
        "selling_price",
        "promotion_price",
        "price_per_unit",
        "promotion_description",
        "pdp_url",
    ]
]

Invalid rows: 0


,unique_id,product_name,regular_price,selling_price,promotion_price,price_per_unit,promotion_description,pdp_url


In [96]:
import pandas as pd

# 1. Identify which IDs are duplicated
duplicate_ids = df[df.duplicated(subset=['unique_id'], keep=False)]

# 2. Count occurrences of each duplicate ID
duplicate_counts = df['unique_id'].value_counts()
duplicate_counts = duplicate_counts[duplicate_counts > 1]


In [97]:
duplicate_ids

,unique_id,competitor_name,store_name,store_addressline1,store_addressline2,store_suburb,store_state,store_postcode,store_addressid,extraction_date,product_name,brand,brand_type,grammage_quantity,grammage_unit,drained_weight,producthierarchy_level1,producthierarchy_level2,producthierarchy_level3,producthierarchy_level4,producthierarchy_level5,producthierarchy_level6,producthierarchy_level7,regular_price,selling_price,price_was,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_type,percentage_discount,promotion_description,package_sizeof_sellingprice,per_unit_sizedescription,price_valid_from,price_per_unit,multi_buy_item_count,multi_buy_items_price_total,currency,breadcrumb,pdp_url,variants,product_description,instructions,storage_instructions,preparationinstructions,instructionforuse,country_of_origin,allergens,age_of_the_product,age_recommendations,flavour,nutritions,nutritional_information,vitamins,labelling,grade,region,packaging,receipies,processed_food,barcode,frozen,chilled,organictype,cooking_part,handmade,max_heating_temperature,special_information,label_information,dimensions,special_nutrition_purpose,feeding_recommendation,warranty,color,model_number,material,usp,dosage_recommendation,tasting_note,food_preservation,size,rating,review,file_name_1,image_url_1,file_name_2,image_url_2,file_name_3,image_url_3,file_name_4,image_url_4,file_name_5,image_url_5,file_name_6,image_url_6,competitor_product_key,fit_guide,occasion,material_composition,style,care_instructions,heel_type,heel_height,upc,features,dietary_lifestyle,manufacturer_address,importer_address,distributor_address,vinification_details,recycling_information,return_address,alchol_by_volume,beer_deg,netcontent,netweight,site_shown_uom,ingredients,random_weight_flag,instock,promo_limit,product_unique_key,multibuy_items_pricesingle,perfect_match,servings_per_pack,warning,suitable_for,standard_drinks,environmental,grape_variety,retail_limit,promotion_description_clean,producthierarchy_level3_clean,producthierarchy_level4_clean,regular_price_num,selling_price_num,promotion_price_num,price_was_num,grammage_unit_norm,site_qty


In [98]:
df.grammage_unit.value_counts()

grammage_unit
g     5589
pc    2716
ml    1693
cl     462
kg     433
l      392
Name: count, dtype: int64

In [99]:
df[df.grammage_unit == "stuck"]

,unique_id,competitor_name,store_name,store_addressline1,store_addressline2,store_suburb,store_state,store_postcode,store_addressid,extraction_date,product_name,brand,brand_type,grammage_quantity,grammage_unit,drained_weight,producthierarchy_level1,producthierarchy_level2,producthierarchy_level3,producthierarchy_level4,producthierarchy_level5,producthierarchy_level6,producthierarchy_level7,regular_price,selling_price,price_was,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_type,percentage_discount,promotion_description,package_sizeof_sellingprice,per_unit_sizedescription,price_valid_from,price_per_unit,multi_buy_item_count,multi_buy_items_price_total,currency,breadcrumb,pdp_url,variants,product_description,instructions,storage_instructions,preparationinstructions,instructionforuse,country_of_origin,allergens,age_of_the_product,age_recommendations,flavour,nutritions,nutritional_information,vitamins,labelling,grade,region,packaging,receipies,processed_food,barcode,frozen,chilled,organictype,cooking_part,handmade,max_heating_temperature,special_information,label_information,dimensions,special_nutrition_purpose,feeding_recommendation,warranty,color,model_number,material,usp,dosage_recommendation,tasting_note,food_preservation,size,rating,review,file_name_1,image_url_1,file_name_2,image_url_2,file_name_3,image_url_3,file_name_4,image_url_4,file_name_5,image_url_5,file_name_6,image_url_6,competitor_product_key,fit_guide,occasion,material_composition,style,care_instructions,heel_type,heel_height,upc,features,dietary_lifestyle,manufacturer_address,importer_address,distributor_address,vinification_details,recycling_information,return_address,alchol_by_volume,beer_deg,netcontent,netweight,site_shown_uom,ingredients,random_weight_flag,instock,promo_limit,product_unique_key,multibuy_items_pricesingle,perfect_match,servings_per_pack,warning,suitable_for,standard_drinks,environmental,grape_variety,retail_limit,promotion_description_clean,producthierarchy_level3_clean,producthierarchy_level4_clean,regular_price_num,selling_price_num,promotion_price_num,price_was_num,grammage_unit_norm,site_qty


In [100]:
duplicates = df[
    df.duplicated(subset=["store_addressline1", "unique_id"], keep=False)
].sort_values(["store_addressline1", "unique_id"])

print("Duplicate unique_id within stores:", len(duplicates))

display(duplicates[["store_addressline1", "unique_id"]])


Duplicate unique_id within stores: 0


,store_addressline1,unique_id


In [101]:
df.dtypes

unique_id                                 int64
competitor_name                          object
store_name                              float64
store_addressline1                      float64
store_addressline2                      float64
store_suburb                            float64
store_state                             float64
store_postcode                          float64
store_addressid                         float64
extraction_date                          object
product_name                             object
brand                                   float64
brand_type                              float64
grammage_quantity                        object
grammage_unit                            object
drained_weight                          float64
producthierarchy_level1                  object
producthierarchy_level2                  object
producthierarchy_level3                  object
producthierarchy_level4                  object
producthierarchy_level5                 

In [102]:
issues = df[
    df["grammage_quantity"].astype(str).str.count(r"\d+\s*x") > 1
][["grammage_quantity", "pdp_url"]].reset_index(names="row_no")

issues

,row_no,grammage_quantity,pdp_url


In [103]:
wine = df[
    df["producthierarchy_level1"].astype(str).str.contains("wein|wine", case=False, na=False)
]

issues = wine[
    wine["site_shown_uom"].astype(str).str.contains(r"\b[2-9]\d*\s*x\b", regex=True, na=False)
][["site_shown_uom", "regular_price", "selling_price", "pdp_url"]]

issues.reset_index(names="row_no")

,row_no,site_shown_uom,regular_price,selling_price,pdp_url


In [104]:
import pandas as pd

issues = []

for idx, row in df.iterrows():

    rp = str(row.get("regular_price", "")).strip()
    sp = str(row.get("selling_price", "")).strip()
    pp = str(row.get("promotion_price", "")).strip()
    pdesc = str(row.get("promotion_description", "")).strip()

    # Ignore NaN represented as string
    if rp.lower() == "nan": rp = ""
    if sp.lower() == "nan": sp = ""
    if pp.lower() == "nan": pp = ""
    if pdesc.lower() == "nan": pdesc = ""

    remarks = []

    # Rule 1: Regular price contains comma
    if "," in rp:
        remarks.append("Regular price contains comma (,) instead of dot (.)")

    # Rule 2: Regular price missing
    if rp == "" and sp != "":
        remarks.append("Regular price missing (should be assigned from selling price)")

    # Rule 3: Promotion description missing but prices exist
    if pdesc == "" and rp != "" and sp != "":
        if pp != sp:
            remarks.append("Promotion description missing - promotion_price should equal selling_price")

    # Rule 4: Aktion products with all prices (reference only)
    if "aktion" in pdesc.lower():
        if rp == "":
            remarks.append("Aktion product with missing regular price (requires previous iteration data)")
        elif pp == "":
            remarks.append("Aktion product missing promotion_price")

    if remarks:
        issues.append({
            "row_no": idx + 1,
            "profile_url": row.get("profile_url", ""),
            "regular_price": rp,
            "selling_price": sp,
            "promotion_price": pp,
            "promotion_description": pdesc,
            "issue": " | ".join(remarks)
        })

issues_df = pd.DataFrame(issues)

print(f"Total issues found: {len(issues_df)}")
display(issues_df)

Total issues found: 7673


,row_no,profile_url,regular_price,selling_price,promotion_price,promotion_description,issue
0,1,,0.55,0.55,,,Promotion description missing - promotion_price should equal selling_price
1,2,,0.31,0.31,,,Promotion description missing - promotion_price should equal selling_price
2,3,,1.75,1.75,,,Promotion description missing - promotion_price should equal selling_price
3,4,,2.15,2.15,,,Promotion description missing - promotion_price should equal selling_price
4,5,,1.99,1.99,,,Promotion description missing - promotion_price should equal selling_price
5,6,,1.95,1.95,,,Promotion description missing - promotion_price should equal selling_price
6,7,,0.45,0.45,,,Promotion description missing - promotion_price should equal selling_price
7,8,,1.95,1.95,,,Promotion description missing - promotion_price should equal selling_price
8,9,,2.89,2.89,,,Promotion description missing - promotion_price should equal selling_price
9,11,,2.49,2.49,,,Promotion description missing - promotion_price should equal selling_price


In [105]:
df[df.selling_price.isna()].pdp_url

Series([], Name: pdp_url, dtype: object)

In [106]:
mask = (
    df['selling_price'].eq(df['regular_price']) &
    df['selling_price'].eq(df['promotion_price'])
)

issues = df.loc[mask, ['pdp_url', 'selling_price', 'regular_price', "promotion_description", 'promotion_price']].copy()
issues['row_no'] = issues.index + 2


issues[['row_no', 'pdp_url', 'selling_price', 'regular_price', "promotion_description", 'promotion_price']]

,row_no,pdp_url,selling_price,regular_price,promotion_description,promotion_price


In [107]:
# Find Mehrweg products that are not yogurt
# and have missing price_per_unit

mehrweg_mask = (
    df['promotion_description']
    .fillna('')
    .astype(str)
    .str.contains('Mehrweg', case=False, na=False)
)

# Exclude yogurt products
not_yogurt_mask = ~df['product_name'].fillna('').astype(str).str.contains(
    'yogurt|joghurt|yoghurt',
    case=False,
    na=False
)

# price_per_unit is missing / empty
price_per_unit_missing = (
    df['price_per_unit'].isna() |
    df['price_per_unit'].astype(str).str.strip().eq('')
)

mask = mehrweg_mask & not_yogurt_mask & price_per_unit_missing

issues = df.loc[
    mask,
    ['pdp_url', 'product_name', 'promotion_description', 'price_per_unit']
].copy()

# Excel row number
issues['row_no'] = issues.index + 2

issues = issues[
    ['row_no', 'pdp_url', 'product_name',
     'promotion_description', 'price_per_unit']
]

print(f"Total Mehrweg issues: {len(issues)}")
print(issues.to_string(index=False))

Total Mehrweg issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, product_name, promotion_description, price_per_unit]
Index: []


In [108]:
# Check if selling_price or promotion_price is greater than regular_price

regular_price = pd.to_numeric(df['regular_price'], errors='coerce')
selling_price = pd.to_numeric(df['selling_price'], errors='coerce')
promotion_price = pd.to_numeric(df['promotion_price'], errors='coerce')

mask = (
    regular_price.notna() &
    (
        (selling_price.notna() & (selling_price > regular_price)) |
        (promotion_price.notna() & (promotion_price > regular_price))
    )
)

issues = df.loc[
    mask,
    ['pdp_url', 'promotion_description',
     'regular_price', 'selling_price', 'promotion_price']
].copy()

# Excel row number
issues['row_no'] = issues.index + 2

issues = issues
    ['row_no', 'pdp_url', 'promotion_description',
     'regular_price', 'selling_price', 'promotion_price']

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

IndentationError: unexpected indent (410566826.py, line 25)

In [109]:
mask = (
    df["regular_price"].notna() &
    df["price_was"].notna() &
    (pd.to_numeric(df["regular_price"], errors="coerce") ==
     pd.to_numeric(df["price_was"], errors="coerce"))
)

issues = df.loc[mask, ["pdp_url", "regular_price", "price_was"]].copy()
issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "regular_price", "price_was"]]

issues

,row_no,pdp_url,regular_price,price_was
68,70,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-irish-bramley-cooking-apples-4-piece-id-1028455001,1.59,1.59
70,72,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-net-oranges-15-kg-id-1889737000,1.95,1.95
103,105,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-irish-mini-royal-gala-apples-9-piece-id-1984345000,1.79,1.79
141,143,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-mixed-peppers-2-piece-id-1024508000,1.69,1.69
155,157,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-irish-closed-cup-mushrooms-300-g-id-2011994000,1.19,1.19
164,166,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-courgettes-2-piece-id-1024916000,1.75,1.75
485,487,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/mccambridge-spelt-soda-bread-500-g-id-1021087002,2.75,2.75
501,503,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/johnston-mooney-obrien-wholemeal-sliced-pan-800-g-id-1019326000,2.25,2.25
505,507,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/johnston-mooney-obrien-wholegrain-sliced-pan-800-g-id-1026079000,2.39,2.39
630,632,https://shop.supervalu.ie/sm/delivery/rsid/5550/product/fitzgeralds-mini-garlic-and-coriander-naan-breads-6-piece-id-1021247000,2.69,2.69


In [110]:
df.organictype.value_counts()

Series([], Name: count, dtype: int64)

In [111]:
for i, ch in enumerate(url):
    if ch == " ":
        print(i, "Normal space")
    elif ch == "\xa0":
        print(i, "NBSP")

NameError: name 'url' is not defined

In [112]:
issues = df[
    df["product_unique_key"].astype(str) != (df["unique_id"].astype(str) + "P")
][["unique_id", "product_unique_key", "pdp_url"]].reset_index(names="row_no")

issues


,row_no,unique_id,product_unique_key,pdp_url


In [113]:
invalid_urls = [u.replace(" ", "\xa0") for u in invalid_urls]

issues = df[df["pdp_url"].isin(invalid_urls)][
    ["unique_id", "pdp_url"]
].reset_index(names="row_no")

issues

,row_no,unique_id,pdp_url


In [114]:
df[df['site_shown_uom'] == ""]


,unique_id,competitor_name,store_name,store_addressline1,store_addressline2,store_suburb,store_state,store_postcode,store_addressid,extraction_date,product_name,brand,brand_type,grammage_quantity,grammage_unit,drained_weight,producthierarchy_level1,producthierarchy_level2,producthierarchy_level3,producthierarchy_level4,producthierarchy_level5,producthierarchy_level6,producthierarchy_level7,regular_price,selling_price,price_was,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_type,percentage_discount,promotion_description,package_sizeof_sellingprice,per_unit_sizedescription,price_valid_from,price_per_unit,multi_buy_item_count,multi_buy_items_price_total,currency,breadcrumb,pdp_url,variants,product_description,instructions,storage_instructions,preparationinstructions,instructionforuse,country_of_origin,allergens,age_of_the_product,age_recommendations,flavour,nutritions,nutritional_information,vitamins,labelling,grade,region,packaging,receipies,processed_food,barcode,frozen,chilled,organictype,cooking_part,handmade,max_heating_temperature,special_information,label_information,dimensions,special_nutrition_purpose,feeding_recommendation,warranty,color,model_number,material,usp,dosage_recommendation,tasting_note,food_preservation,size,rating,review,file_name_1,image_url_1,file_name_2,image_url_2,file_name_3,image_url_3,file_name_4,image_url_4,file_name_5,image_url_5,file_name_6,image_url_6,competitor_product_key,fit_guide,occasion,material_composition,style,care_instructions,heel_type,heel_height,upc,features,dietary_lifestyle,manufacturer_address,importer_address,distributor_address,vinification_details,recycling_information,return_address,alchol_by_volume,beer_deg,netcontent,netweight,site_shown_uom,ingredients,random_weight_flag,instock,promo_limit,product_unique_key,multibuy_items_pricesingle,perfect_match,servings_per_pack,warning,suitable_for,standard_drinks,environmental,grape_variety,retail_limit,promotion_description_clean,producthierarchy_level3_clean,producthierarchy_level4_clean,regular_price_num,selling_price_num,promotion_price_num,price_was_num,grammage_unit_norm,site_qty


In [330]:
import pandas as pd

issues = []

for col in df.select_dtypes(include="object").columns:

    mask = df[col].fillna("").astype(str).str.contains(
        r'^\s+|\s+$|\t|\n|\r| {2,}',
        regex=True,
        na=False
    )

    if mask.any():
        temp = df.loc[mask, [col, "pdp_url"]].copy()

        temp["row_no"] = temp.index + 2
        temp["column"] = col
        temp["issue_value"] = temp[col]

        issues.append(
            temp[["row_no", "pdp_url", "column", "issue_value"]]
        )

if issues:
    result = pd.concat(issues, ignore_index=True)

    print(f"Total whitespace issues: {len(result)}")
    print(result.to_string(index=False))
else:
    print("No unwanted whitespace found.")

No unwanted whitespace found.


In [331]:
df.grammage_quantity.value_counts()

grammage_quantity
1        1411
500       783
250       721
200       602
100       570
0.75      523
400       480
150       437
300       326
0.5       284
0.7       254
125       218
0.33      175
1 000     143
180       142
330       141
50        136
20        130
1.5       129
350       120
450       117
75        115
750       114
80        106
0.25      103
2          92
160        90
120        89
800        83
175        80
600        80
40         76
190        75
220        73
70         73
360        73
90         72
140        71
60         64
130        58
30         57
0.2        54
10         54
240        54
110        54
340        53
170        52
700        51
85         50
225        45
25         42
280        40
3          40
95         38
45         36
375        36
320        36
230        35
270        34
650        34
210        32
185        31
65         31
370        29
460        28
4          28
670        28
55         25
420        25
6          24
15

In [115]:
import pandas as pd
import re

issues = []

# Check all text/object columns
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    s = df[col].fillna("").astype(str)

    # 1. Value ending with comma
    mask_comma = s.str.rstrip().str.endswith(",")

    if mask_comma.any():
        temp = df.loc[mask_comma, [col]].copy()
        temp["row_no"] = temp.index + 2
        temp["column"] = col
        temp["issue"] = "Value ends with comma (,)"
        temp["issue_value"] = temp[col]

        # Add PDP URL if available
        if "pdp_url" in df.columns:
            temp["URL"] = df.loc[mask_comma, "pdp_url"]

        issues.append(
            temp[["row_no", "URL", "column", "issue_value", "issue"]]
        )

    # 2. Leading/trailing whitespace
    mask_space = s.ne(s.str.strip()) & s.ne("")

    if mask_space.any():
        temp = df.loc[mask_space, [col]].copy()
        temp["row_no"] = temp.index + 2
        temp["column"] = col
        temp["issue"] = "Leading/trailing whitespace"
        temp["issue_value"] = temp[col]

        if "pdp_url" in df.columns:
            temp["URL"] = df.loc[mask_space, "pdp_url"]

        issues.append(
            temp[["row_no", "URL", "column", "issue_value", "issue"]]
        )

# Combine results
if issues:
    result = pd.concat(issues, ignore_index=True)

    print(result.to_string(index=False))
else:
    print("No issues found.")

 row_no                                                                                                                                       URL               column                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [116]:
mask = (
    df["product_name"].astype("string").str.startswith("Denner", na=False)
    & df["brand"].astype("string").str.strip().ne("Denner")
)

result = df.loc[mask, ["unique_id", "pdp_url", "product_name", "brand"]].copy()
result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[["row_no", "unique_id", "pdp_url", "product_name", "brand"]].to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, unique_id, pdp_url, product_name, brand]
Index: []


In [117]:
mask = (
    df["file_name"].notna()
    & df["file_name"].astype(str).str.strip().ne("")
)

result = df.loc[mask, ["unique_id", "pdp_url", "file_name"]].copy()
result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[["row_no", "unique_id", "pdp_url", "file_name"]].to_string(index=False))

KeyError: 'file_name'

In [118]:
price_cols = ["regular_price"]

for col in price_cols:
    # Check exactly 2 decimal places
    mask = (
        df[col].notna() &
        ~df[col].astype(str).str.match(r'^\d+\.\d{2}$')
    )

    if mask.any():
        issues = df.loc[mask, ['pdp_url', col]].copy()

        # Row number = dataframe index + 2
        issues.insert(0, 'row_no', issues.index + 2)

        print(f"\n=== {col} - Invalid 2 Decimal Format ===")
        print(issues.to_string(index=False))
        print(f"\nTotal issues: {len(issues)}")

    else:
        print(f"{col}: All values are in 2 decimal format.")


=== regular_price - Invalid 2 Decimal Format ===
 row_no                                                                                                                                                       pdp_url  regular_price
     11                                            https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-blueberries-225-g-id-1683185003            5.0
     13                                                 https://shop.supervalu.ie/sm/delivery/rsid/5550/product/keelings-strawberries-family-pack-350-g-id-1386984000            4.5
     23                                                https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-xl-mangos-ready-to-eat-2-piece-id-1998084000            4.0
     24                                                https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-plums-4-piece-id-1331123001            3.0
     27                                      

In [119]:
# Promotion present but both promotion dates are missing

promotion_present = (
    df["promotion_description"].fillna("").astype(str).str.strip().ne("") |
    df["promotion_price"].notna()
)

dates_missing = (
    df["promotion_valid_from"].fillna("").astype(str).str.strip().eq("") &
    df["promotion_valid_upto"].fillna("").astype(str).str.strip().eq("")
)

mask = promotion_present & dates_missing

issues = df.loc[
    mask,
    [
        "promotion_description",
        "promotion_price",
        "promotion_valid_from",
        "promotion_valid_upto",
        "pdp_url"
    ]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    [
        "row_no",
        "pdp_url",
        "promotion_description",
        "promotion_price",
        "promotion_valid_from",
        "promotion_valid_upto"
    ]
]

print(f"Total issues: {len(issues)}")
print(issues.head(5).to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, promotion_price, promotion_valid_from, promotion_valid_upto]
Index: []


In [120]:
# Check whether price_was is the same as regular_price
# If they are different, the record is reported as an issue.

mask = (
    df['price_was'].notna() &
    df['regular_price'].notna() &
    (df['price_was'].astype(str) != df['regular_price'].astype(str))
)

issues = df.loc[
    mask,
    ['pdp_url', 'price_was', 'regular_price']
].copy()

# Add Excel row number (+2 because row 1 is the header)
issues['row_no'] = issues.index + 2

# Reorder columns
issues = issues[['row_no', 'pdp_url', 'price_was', 'regular_price']]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, price_was, regular_price]
Index: []


In [121]:
# Convert both columns to numeric values
price_was = pd.to_numeric(df["price_was"], errors="coerce")
regular_price = pd.to_numeric(df["regular_price"], errors="coerce")

# Find rows where price_was and regular_price are NOT the same
mask = (
    price_was.notna() &
    regular_price.notna() &
    (price_was != regular_price)
)

issues = df.loc[
    mask,
    ["pdp_url", "price_was", "regular_price"]
].copy()

# Excel row number
issues["row_no"] = issues.index + 2

# Arrange columns
issues = issues[
    ["row_no", "pdp_url", "price_was", "regular_price"]
]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, price_was, regular_price]
Index: []


In [122]:
import pandas as pd
import re

# Extract the unit from price_per_unit
df["price_unit"] = (
    df["price_per_unit"]
    .astype("string")
    .str.extract(
        r'(?:per\s+\d+(?:[.,]\d+)?\s+)?([a-zA-Z]+)',
        flags=re.IGNORECASE,
        expand=False
    )
)

# Normalize both units
df["grammage_unit_clean"] = (
    df["grammage_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["price_unit_clean"] = (
    df["price_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Find rows where units do NOT match
mismatch = df[
    df["grammage_unit_clean"].notna() &
    df["price_unit_clean"].notna() &
    (df["grammage_unit_clean"] != df["price_unit_clean"])
]

# Display relevant columns
print(mismatch[
    ["grammage_unit", "price_per_unit", "price_unit", "pdp_url"]
].to_string(index=False))

print("Total unit mismatch:", len(mismatch))

Empty DataFrame
Columns: [grammage_unit, price_per_unit, price_unit, pdp_url]
Index: []
Total unit mismatch: 0


In [123]:
import pandas as pd
import re

# Extract the unit from price_per_unit
df["price_unit"] = (
    df["price_per_unit"]
    .astype("string")
    .str.extract(
        r'(?:per\s+\d+(?:[.,]\d+)?\s+)?([a-zA-Z]+)',
        flags=re.IGNORECASE,
        expand=False
    )
)

# Normalize both units
df["grammage_unit_clean"] = (
    df["grammage_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["price_unit_clean"] = (
    df["price_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Find rows where units do NOT match
mismatch = df[
    df["grammage_unit_clean"].notna() &
    df["price_unit_clean"].notna() &
    (df["grammage_unit_clean"] != df["price_unit_clean"])
].copy()

# Add row number (+2 because row 1 is the header)
mismatch["row_no"] = mismatch.index + 2

# Display relevant columns
result = mismatch[
    ["row_no", "grammage_unit", "price_per_unit", "price_unit", "pdp_url"]
]

print(result.to_string(index=False))

print(f"\nTotal unit mismatch: {len(result)}")

Empty DataFrame
Columns: [row_no, grammage_unit, price_per_unit, price_unit, pdp_url]
Index: []

Total unit mismatch: 0


In [124]:
# Convert grammage_quantity to numeric
quantity = pd.to_numeric(df["grammage_quantity"], errors="coerce")

# Clean grammage unit
unit = (
    df["grammage_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Find quantity values below 1 but unit is still gram
mask = (
    quantity.notna() &
    (quantity < 1) &
    (unit == "g")
)

issues = df.loc[
    mask,
    ["grammage_quantity", "grammage_unit", "pdp_url"]
].copy()

# Excel row number
issues["row_no"] = issues.index + 2

# Arrange columns
issues = issues[
    ["row_no", "pdp_url", "grammage_quantity", "grammage_unit"]
]

print(f"Total grammage conversion issues: {len(issues)}")
print(issues.to_string(index=False))

Total grammage conversion issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, grammage_quantity, grammage_unit]
Index: []


In [125]:
import pandas as pd

quantity = pd.to_numeric(
    df["grammage_quantity"],
    errors="coerce"
)

unit = (
    df["grammage_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Suspicious quantity/unit combinations
mask = (
    ((unit == "g") & (quantity < 1)) |
    ((unit == "kg") & (quantity >= 1000)) |
    ((unit == "ml") & (quantity < 1)) |
    ((unit == "l") & (quantity >= 1000)) |
    ((unit == "mg") & (quantity >= 1000))
)

issues = df.loc[
    mask,
    ["grammage_quantity", "grammage_unit", "pdp_url"]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    ["row_no", "pdp_url", "grammage_quantity", "grammage_unit"]
]

print(f"Total suspicious quantity/unit issues: {len(issues)}")
print(issues.to_string(index=False))

Total suspicious quantity/unit issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, grammage_quantity, grammage_unit]
Index: []


In [126]:
import pandas as pd

# Words that should be treated as invalid/missing text
invalid_words = [
    "null",
    "none",
    "empty",
    "n/a",
    "na",
    "nan",
    "nil",
    "missing",
    "undefined"
]

issues = []

for col in df.columns:
    # Check only object/string columns
    if df[col].dtype == "object":
        
        mask = (
            df[col]
            .astype("string")
            .str.strip()
            .str.lower()
            .isin(invalid_words)
        )

        if mask.any():
            temp = df.loc[mask, [col, "pdp_url"]].copy()
            temp["row_no"] = temp.index + 2
            temp["column"] = col
            temp["issue_value"] = temp[col]

            issues.append(
                temp[["row_no", "pdp_url", "column", "issue_value"]]
            )

if issues:
    result = pd.concat(issues, ignore_index=True)

    print(f"Total invalid missing-value text records: {len(result)}")
    print(result.to_string(index=False))
else:
    print("No 'null', 'none', 'empty', 'N/A', etc. text values found.")

No 'null', 'none', 'empty', 'N/A', etc. text values found.


In [127]:
issues = df[
    (
        df['promotion_price'].isna() &
        df['promotion_type'].isna() &
        df['percentage_discount'].isna()
    )
    &
    (
        df['regular_price'].isna() |
        df['selling_price'].isna()
    )
].copy()

issues.insert(0, 'row_no', issues.index + 2)

issues[['row_no', 'pdp_url', 'regular_price', 'selling_price',
        'promotion_price', 'promotion_type', 'percentage_discount']]

,row_no,pdp_url,regular_price,selling_price,promotion_price,promotion_type,percentage_discount


In [128]:
import pandas as pd

# Convert prices to numeric
regular_price = pd.to_numeric(df["regular_price"], errors="coerce")
selling_price = pd.to_numeric(df["selling_price"], errors="coerce")
promotion_price = pd.to_numeric(df["promotion" \
"_price"], errors="coerce")

# Check where regular_price exists but promotion_price != selling_price
mask = (
    regular_price.notna() &
    selling_price.notna() &
    (
        promotion_price.isna() |
        (promotion_price != selling_price)
    )
)

issues = df.loc[
    mask,
    [
        "pdp_url",
        "regular_price",
        "selling_price",
        "promotion_price"
    ]
].copy()

# Add row number
issues["row_no"] = issues.index + 2

# Arrange columns
issues = issues[
    [
        "row_no",
        "pdp_url",
        "regular_price",
        "selling_price",
        "promotion_price"
    ]
]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

Total issues: 8716
 row_no                                                                                                                                                       pdp_url  regular_price  selling_price  promotion_price
      2                                                https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-loose-pink-lady-apples-1-piece-id-1179723000           0.55           0.55              NaN
      3                                                      https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-single-loose-banana-1-kg-id-1013680000           0.31           0.31              NaN
      4                                                                https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-broccoli-350-g-id-1030326000           1.75           1.75              NaN
      5                                             https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-organic-fai

In [129]:
import pandas as pd

# Function to identify non-empty values
def is_present(s):
    return s.notna() & s.astype(str).str.strip().ne('')


# -------------------------------
# 1. Promotion description present
# -------------------------------

promo = is_present(df['promotion_description'])


# -------------------------------
# 2. Check price fields
# -------------------------------

price_cols = [
    'promotion_price',
    'price_was',
    'selling_price',
    'regular_price'
]

price_count = df[price_cols].apply(is_present).sum(axis=1)


# -------------------------------
# RULE 1
# Promotion description present
# but promotion_price != selling_price
# -------------------------------

promotion_price = pd.to_numeric(
    df['promotion_price'], errors='coerce'
)

selling_price = pd.to_numeric(
    df['selling_price'], errors='coerce'
)

rule1 = (
    promo &
    (
        df['promotion_price'].isna() |
        df['selling_price'].isna() |
        (promotion_price != selling_price)
    )
)


# -------------------------------
# RULE 2
# Only ONE price is present
# but promotion description is present
# -------------------------------

rule2 = promo & (price_count == 1)


# -------------------------------
# Combine violations
# -------------------------------

violations = df[rule1 | rule2].copy()


# -------------------------------
# Add row number from index
# -------------------------------

violations.insert(
    0,
    'row_no',
    violations.index
)


# -------------------------------
# Add issue reason

# -------------------------------

violations['issue'] = ''

violations.loc[
    rule1[violations.index],
    'issue'
] += 'Promotion price/selling price mismatch; '

violations.loc[
    rule2[violations.index],
    'issue'
] += 'Only one price present but promotion description populated; '


# -------------------------------
# Final output
# -------------------------------

result = violations[
    [
        'row_no',
        'pdp_url',
        'promotion_description',
        'promotion_price',
        'price_was',
        'selling_price',
        'regular_price',
        'issue'
    ]
]

print("Total violating products:", len(result))

print(result.to_string(index=False))

Total violating products: 1043
 row_no                                                                                                                                                    pdp_url           promotion_description  promotion_price  price_was  selling_price  regular_price                                    issue
      9                                         https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-blueberries-225-g-id-1683185003                       3 for €10              NaN        NaN           5.00           5.00 Promotion price/selling price mismatch; 
     11                                              https://shop.supervalu.ie/sm/delivery/rsid/5550/product/keelings-strawberries-family-pack-350-g-id-1386984000                       3 for €10              NaN        NaN           4.50           4.50 Promotion price/selling price mismatch; 
     21                                             https://shop.supervalu.ie/sm/delive

In [130]:
mask = (
    df["price_valid_from"].fillna("").astype(str).str.strip()
    !=
    df["promotion_valid_from"].fillna("").astype(str).str.strip()
)

result = df.loc[mask, ["pdp_url", "price_valid_from", "promotion_valid_from"]].copy()
result["row_no"] = result.index + 2

print("Total mismatches:", len(result))
print(result[["row_no", "pdp_url", "price_valid_from", "promotion_valid_from"]].to_string(index=False))

Total mismatches: 0
Empty DataFrame
Columns: [row_no, pdp_url, price_valid_from, promotion_valid_from]
Index: []


In [131]:
df.dtypes

unique_id                                 int64
competitor_name                          object
store_name                              float64
store_addressline1                      float64
store_addressline2                      float64
store_suburb                            float64
store_state                             float64
store_postcode                          float64
store_addressid                         float64
extraction_date                          object
product_name                             object
brand                                   float64
brand_type                              float64
grammage_quantity                        object
grammage_unit                            object
drained_weight                          float64
producthierarchy_level1                  object
producthierarchy_level2                  object
producthierarchy_level3                  object
producthierarchy_level4                  object
producthierarchy_level5                 

In [132]:
mask = (
    df["percentage_discount"]
    .fillna("")
    .astype(str)
    .str.contains(r"[%\-]", regex=True)
)

result = df.loc[
    mask,
    ["pdp_url", "percentage_discount"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[["row_no", "pdp_url", "percentage_discount"]].to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, percentage_discount]
Index: []


In [133]:
mask = (
    df["regular_price"].fillna("").astype(str).str.strip()
    !=
    df["price_was"].fillna("").astype(str).str.strip()
)

result = df.loc[
    mask,
    ["pdp_url", "regular_price", "price_was"]
].copy()

result["row_no"] = result.index + 2

print("Total mismatches:", len(result))
print(result[["row_no", "pdp_url", "regular_price", "price_was"]].to_string(index=False))

Total mismatches: 8716
 row_no                                                                                                                                                       pdp_url  regular_price  price_was
      2                                                https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-loose-pink-lady-apples-1-piece-id-1179723000           0.55        NaN
      3                                                      https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-single-loose-banana-1-kg-id-1013680000           0.31        NaN
      4                                                                https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-broccoli-350-g-id-1030326000           1.75        NaN
      5                                             https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-organic-fairtrade-bananas-5-piece-id-1085596000           2.15        NaN
      6          

In [134]:
df[df.image_url_1.isna()].pdp_url

5835    https://shop.supervalu.ie/sm/delivery/rsid/5550/product/silvermints-cool-sugar-free-mints-32-g-id-2026823000
Name: pdp_url, dtype: object

In [133]:
# Promotion description and promotion price are present,
# but promotion dates are missing

mask = (
    df["promotion_description"].fillna("").astype(str).str.strip().ne("")
    & df["promotion_price"].fillna("").astype(str).str.strip().ne("")
    & (
        df["promotion_valid_from"].fillna("").astype(str).str.strip().eq("")
        | df["promotion_valid_upto"].fillna("").astype(str).str.strip().eq("")
    )
)

result = df.loc[
    mask,
    ["pdp_url", "promotion_description", "promotion_price",
     "pr"
     "omotion_valid_from", "promotion_valid_upto"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[[
    "row_no", "pdp_url", "promotion_description", "promotion_price",
    "promotion_valid_from", "promotion_valid_upto"
]].tail(10).to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, promotion_price, promotion_valid_from, promotion_valid_upto]
Index: []


In [135]:
# Promotion description and promotion price are present,
# but BOTH promotion dates are missing

mask = (
    df["promotion_description"].fillna("").astype(str).str.strip().ne("")
    & df["promotion_price"].fillna("").astype(str).str.strip().ne("")
    & (
        df["promotion_valid_from"].fillna("").astype(str).str.strip().eq("")
        & df["promotion_valid_upto"].fillna("").astype(str).str.strip().eq("")
    )
)

result = df.loc[
    mask,
    ["pdp_url", "promotion_description", "promotion_price",
     "promotion_valid_from", "promotion_valid_upto"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(
    result[[
        "row_no", "pdp_url", "promotion_description", "promotion_price",
        "promotion_valid_from", "promotion_valid_upto"
    ]].tail(10).to_string(index=False)
)

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, promotion_price, promotion_valid_from, promotion_valid_upto]
Index: []


In [136]:
# Check regular_price and selling_price for values
# that are not in exactly two-decimal format

for col in ["regular_price", "selling_price"]:
    price = df[col].fillna("").astype(str).str.strip()

    mask = price.ne("") & ~price.str.fullmatch(r"\d+\.\d{2}", na=False)

    result = df.loc[mask, ["pdp_url", col]].copy()
    result["row_no"] = result.index + 2

    print(f"\nInvalid {col}: {len(result)}")
    print(result[["row_no", "pdp_url", col]].head(5).to_string(index=False))


Invalid regular_price: 4180
 row_no                                                                                                                 pdp_url  regular_price
     11      https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-blueberries-225-g-id-1683185003            5.0
     13           https://shop.supervalu.ie/sm/delivery/rsid/5550/product/keelings-strawberries-family-pack-350-g-id-1386984000            4.5
     23          https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-xl-mangos-ready-to-eat-2-piece-id-1998084000            4.0
     24          https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-plums-4-piece-id-1331123001            3.0
     27 https://shop.supervalu.ie/sm/delivery/rsid/5550/product/supervalu-signature-tastes-kiwi-gold-tray-5-piece-id-1892011000            4.0

Invalid selling_price: 5021
 row_no                                                                             

In [137]:
import pandas as pd

from_date = pd.to_datetime(
    df["promotion_valid_from"],
    errors="coerce",
    dayfirst=True
)

upto_date = pd.to_datetime(
    df["promotion_valid_upto"],
    errors="coerce",
    dayfirst=True
)

mask = (
    from_date.notna()
    & upto_date.notna()
    & (from_date > upto_date)
)

result = df.loc[
    mask,
    ["pdp_url", "promotion_valid_from", "promotion_valid_upto"]
].copy()

result["row_no"] = result.index + 2

print("Total date errors:", len(result))
print(
    result[
        ["row_no", "pdp_url", "promotion_valid_from", "promotion_valid_upto"]
    ].to_string(index=False)
)

Total date errors: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_from, promotion_valid_upto]
Index: []


In [138]:
# Promotion price exists, but regular_price != price_was

mask = (
    df["promotion_price_num"].notna()
    & df["regular_price_num"].notna()
    & df["price_was_num"].notna()
    & df["regular_price_num"].ne(df["price_was_num"])
)

result = df.loc[
    mask,
    ["unique_id", "pdp_url", "regular_price", "price_was", "promotion_price"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[
    ["row_no", "unique_id", "pdp_url", "regular_price", "price_was", "promotion_price"]
].to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, unique_id, pdp_url, regular_price, price_was, promotion_price]
Index: []


In [139]:
mask = (
    df["regular_price_num"].notna()
    & df["selling_price_num"].notna()
    & df["regular_price_num"].lt(df["selling_price_num"])
)

result = df.loc[
    mask,
    ["unique_id", "pdp_url", "regular_price", "selling_price"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[
    ["row_no", "unique_id", "pdp_url", "regular_price", "selling_price"]
].to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, unique_id, pdp_url, regular_price, selling_price]
Index: []


In [140]:
mask = (
    df["promotion_price_num"].notna()
    & df["regular_price_num"].notna()
    & df["promotion_price_num"].gt(df["regular_price_num"])
)

result = df.loc[
    mask,
    ["unique_id", "pdp_url", "regular_price", "promotion_price"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[
    ["row_no", "unique_id", "pdp_url", "regular_price", "promotion_price"]
].to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, unique_id, pdp_url, regular_price, promotion_price]
Index: []


In [141]:
# Check invalid promotion dates
# promotion_valid_from > promotion_valid_upto

df["promotion_valid_from"] = pd.to_datetime(
    df["promotion_valid_from"], errors="coerce"
)

df["promotion_valid_upto"] = pd.to_datetime(
    df["promotion_valid_upto"], errors="coerce"
)

mask = (
    df["promotion_valid_from"].notna()
    & df["promotion_valid_upto"].notna()
    & (df["promotion_valid_from"] > df["promotion_valid_upto"])
)

result = df.loc[
    mask,
    ["unique_id", "pdp_url", "promotion_valid_from", "promotion_valid_upto"]
].copy()

result["row_no"] = result.index + 2

print("Total invalid promotion date records:", len(result))
print(
    result[
        ["row_no", "unique_id", "pdp_url", "promotion_valid_from", "promotion_valid_upto"]
    ].to_string(index=False)
)

Total invalid promotion date records: 5
 row_no  unique_id                                                                                          pdp_url promotion_valid_from promotion_valid_upto
   1174 1028959000 https://shop.supervalu.ie/sm/delivery/rsid/5550/product/loose-haddock-fillets-1-kg-id-1028959000           2026-11-05           2026-08-11
   5887 1561584001   https://shop.supervalu.ie/sm/delivery/rsid/5550/product/cadbury-heroes-tub-475-g-id-1561584001           2026-11-05           2026-09-11
   5895 1576998001    https://shop.supervalu.ie/sm/delivery/rsid/5550/product/cadbury-roses-tub-475-g-id-1576998001           2026-11-05           2026-09-11
   5898 1610789002     https://shop.supervalu.ie/sm/delivery/rsid/5550/product/celebrations-tub-500-g-id-1610789002           2026-11-05           2026-09-11
   5953 1498366002   https://shop.supervalu.ie/sm/delivery/rsid/5550/product/quality-street-tub-550-g-id-1498366002           2026-11-05           2026-09-11


/tmp/ipykernel_74904/16813453.py:4: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["promotion_valid_from"] = pd.to_datetime(


In [142]:
import pandas as pd

# Convert price columns to numeric
price_cols = ["regular_price", "selling_price", "promotion_price"]

for col in price_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Regular price is lower than selling price or promotion price
mask = (
    (df["selling_price"].notna() & (df["regular_price"] < df["selling_price"]))
    | (df["promotion_price"].notna() & (df["regular_price"] < df["promotion_price"]))
)

result = df.loc[
    mask,
    ["unique_id", "pdp_url", "regular_price", "selling_price", "promotion_price"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(
    result[
        ["row_no", "unique_id", "pdp_url", "regular_price", "selling_price", "promotion_price"]
    ].to_string(index=False)
)

Total issues: 0
Empty DataFrame
Columns: [row_no, unique_id, pdp_url, regular_price, selling_price, promotion_price]
Index: []


In [143]:
import pandas as pd

# Check where all three prices are equal
price_cols = ["regular_price", "selling_price", "promotion_price"]

prices = df[price_cols].apply(pd.to_numeric, errors="coerce")

mask = (
    prices.notna().all(axis=1)
    & prices["regular_price"].eq(prices["selling_price"])
    & prices["selling_price"].eq(prices["promotion_price"])
)

result = df.loc[mask, ["pdp_url"] + price_cols].copy()
result.insert(0, "row_no", result.index + 2)

print("Total records with all prices equal:", len(result))

print(result.to_string(index=False))

Total records with all prices equal: 0
Empty DataFrame
Columns: [row_no, pdp_url, regular_price, selling_price, promotion_price]
Index: []


In [144]:
df.grammage_unit.value_counts()

grammage_unit
g     5589
pc    2716
ml    1693
cl     462
kg     433
l      392
Name: count, dtype: int64

In [145]:
df.site_shown_uom.value_counts()

site_shown_uom
Barry's Tea Original Blend 80 Bags (250 g)                                                     3
Listerine Cool Mint Mouthwash (250 ml)                                                         3
Woman (1 Piece)                                                                                2
Hale Strawberry E-Liquid 10ml (10 ml)                                                          2
Barry's Tea Decaffinated 80 Bags (250 g)                                                       2
SuperValu Strawberries (227 g)                                                                 2
Tayto Variety Crisps 6 Pack (25 g)                                                             2
Sistema Storage Container (1 Piece)                                                            2
The Ned Sauvignon Blanc (75 cl)                                                                2
Superquinn Pork Sausages (480 g)                                                               2
Hale Blueberry 